#### This document is for analysis on FACT-E emotional well being (EWB).

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import re

# Preliminary

In [2]:
# Read the CSV file - linked file 
file_path = "Merged_TSQIC_REDCap_ACCESS.xlsx" 
df = pd.read_excel(file_path)
df

,id,operation_date,redcap_event_name,qol_date,age_diagnosis,dob,overall_primary_tumour,overall_regional_ln,overall_distant_metastasis,neotx___notx,...,a_e5,a_e6,a_e7,a_c6,a_c2,a_act11,readmission_30d,postop_comp,los,DischargeDate
0,1,NaT,baseline_arm_1,NaT,NaN,1949-08-04,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,2012-04-27
1,1,2011-08-26,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2011-08-26
2,1,2010-02-20,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,2010-03-08
3,1,2009-05-27,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2009-05-27
4,1,2009-02-25,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0,2009-03-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11964,1788,NaT,baseline_arm_1,NaT,NaN,1947-03-23,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
11965,1788,NaT,baseline_arm_1,2025-06-09,NaN,NaT,NaN,NaN,NaN,NaN,...,0.0,2.0,0.0,2.0,2.0,0.0,NaN,NaN,NaN,NaT
11966,1788,NaT,baseline_arm_1,NaT,NaN,NaT,3,1,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
11967,1789,NaT,baseline_arm_1,NaT,NaN,1945-08-19,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT


In [3]:
def preprocess_patient_data(df):
    """
    Preprocess patient data with expectation_treatment filtering:
    Filter out patients with expectation_treatment != 1 (if information is available)
    """
    
    # Make a copy to avoid modifying original dataframe
    df_processed = df.copy()
    
    # Step 1: expectation_treatment filtering (keep only patients with expectation_treatment == 1)
    print("Step 1: Filtering patients by expectation_treatment...")
    
    # Group by patient id again for expectation_treatment check
    patient_groups = df_processed.groupby('id')
    patients_to_keep = []
    
    for patient_id, group in patient_groups:
        keep_patient = True
        
        # Check if expectation_treatment column exists and has data for this patient
        if 'expectation_treatment' in group.columns:
            expectation_available = group['expectation_treatment'].notna().any()
            
            if expectation_available:
                # Check if any row has expectation_treatment == 1
                has_expectation_treatment = (group['expectation_treatment'] == 1).any()
                
                if not has_expectation_treatment:
                    keep_patient = False
                    print(f"Patient {patient_id}: Filtered out (no expectation_treatment == 1)")
        
        # If patient should be kept, add all their rows
        if keep_patient:
            patients_to_keep.extend(group.index.tolist())
    
    # Filter dataframe to keep only selected patients
    df_processed = df_processed.loc[patients_to_keep]
    print(f"After expectation_treatment filtering: {len(df_processed)} rows remaining")
    
    return df_processed

# Optional: Function to get summary statistics
def get_filtering_summary(df_original, df_filtered):
    """
    Print summary of filtering results
    """
    original_patients = df_original['id'].nunique()
    filtered_patients = df_filtered['id'].nunique()
    original_rows = len(df_original)
    filtered_rows = len(df_filtered)
    
    print(f"\n=== Filtering Summary ===")
    print(f"Original patients: {original_patients}")
    print(f"Filtered patients: {filtered_patients}")
    print(f"Patients removed: {original_patients - filtered_patients}")
    print(f"Original rows: {original_rows}")
    print(f"Filtered rows: {filtered_rows}")
    print(f"Rows removed: {original_rows - filtered_rows}")

# Example usage:
df_filtered = preprocess_patient_data(df)
get_filtering_summary(df, df_filtered)

Step 1: Filtering patients by expectation_treatment...
Patient 8: Filtered out (no expectation_treatment == 1)
Patient 59: Filtered out (no expectation_treatment == 1)
Patient 1179: Filtered out (no expectation_treatment == 1)
After expectation_treatment filtering: 11940 rows remaining

=== Filtering Summary ===
Original patients: 1688
Filtered patients: 1685
Patients removed: 3
Original rows: 11969
Filtered rows: 11940
Rows removed: 29


In [4]:
df = df_filtered.copy()
df

,id,operation_date,redcap_event_name,qol_date,age_diagnosis,dob,overall_primary_tumour,overall_regional_ln,overall_distant_metastasis,neotx___notx,...,a_e5,a_e6,a_e7,a_c6,a_c2,a_act11,readmission_30d,postop_comp,los,DischargeDate
0,1,NaT,baseline_arm_1,NaT,NaN,1949-08-04,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,2012-04-27
1,1,2011-08-26,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2011-08-26
2,1,2010-02-20,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,2010-03-08
3,1,2009-05-27,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2009-05-27
4,1,2009-02-25,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0,2009-03-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11964,1788,NaT,baseline_arm_1,NaT,NaN,1947-03-23,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
11965,1788,NaT,baseline_arm_1,2025-06-09,NaN,NaT,NaN,NaN,NaN,NaN,...,0.0,2.0,0.0,2.0,2.0,0.0,NaN,NaN,NaN,NaT
11966,1788,NaT,baseline_arm_1,NaT,NaN,NaT,3,1,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
11967,1789,NaT,baseline_arm_1,NaT,NaN,1945-08-19,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT


# Preliminary 2

In [5]:
df['id'].nunique()

1685

In [6]:
df['redcap_event_name'].value_counts()

redcap_event_name
baseline_arm_1           3862
surgery_arm_1            3831
preoperative_arm_1       1755
1_month_postop_arm_1      903
1_year_postop_arm_1       350
6_months_postop_arm_1     327
3_months_postop_arm_1     256
Name: count, dtype: int64

# Analysis

In [7]:
# Step 1: Find eligible IDs (unchanged)
def find_eligible_ids(df):
    id_events = df.groupby('id')['redcap_event_name'].apply(set)
    eligible_ids = []
    for id_val, events in id_events.items():
        has_baseline = "baseline_arm_1" in events
        has_preop = "preoperative_arm_1" in events
        has_one_year = "1_year_postop_arm_1" in events
        if (has_baseline and has_one_year) or (has_preop and has_one_year):
            eligible_ids.append(id_val)
    return eligible_ids

# Step 2: Filter dataset (unchanged)
def filter_eligible_data(df, eligible_ids):
    return df[df['id'].isin(eligible_ids)].copy()

# Step 3: Calculate all subscale scores (unchanged)
def calculate_subscale_score(df, cols, reverse_items=None, subscale_name=""):
    """Calculate subscale scores according to FACT scoring guidelines, only if >=50% items are answered"""
    # Convert to numeric and handle reverse scoring
    for col in cols:
        df[f"{col}_score"] = pd.to_numeric(df[col], errors='coerce')
        if reverse_items and col in reverse_items:
            df[f"{col}_score"] = 4 - df[f"{col}_score"]
    
    # Calculate the number of items and the 50% threshold
    total_items = len(cols)
    min_items_required = total_items / 2  # 50% threshold
    
    # Count non-NaN items for each row
    score_cols = [f"{col}_score" for col in cols]
    num_answered_items = df[score_cols].notna().sum(axis=1)
    
    # Calculate subscale score only if >=50% items are answered
    df[f"{subscale_name}_subscale_score"] = np.where(
        num_answered_items >= min_items_required,
        (df[score_cols].sum(axis=1, skipna=True) * total_items) / 
        df[score_cols].notna().sum(axis=1),
        np.nan
    )
    return df

def calculate_all_scores(df):
    pwb_cols = [f"gp{i}" for i in range(1, 8)]
    swb_cols = [f"gs{i}" for i in range(1, 8)]
    ewb_cols = [f"ge{i}" for i in range(1, 7)]
    fwb_cols = [f"gf{i}" for i in range(1, 8)]
    ecs_cols = [f"a_hn{i}" for i in range(1, 6)] + ["a_hn7", "a_hn10"] + \
                [f"a_e{i}" for i in range(1, 8)] + ["a_c6", "a_c2", "a_act11"]

    pwb_reverse = pwb_cols
    ewb_reverse = [f"ge{i}" for i in range(1, 7) if i != 2]
    ecs_reverse = ["a_e1", "a_e2", "a_e3", "a_e4", "a_e5", "a_e7", "a_act11", 
                    "a_c2", "a_hn2", "a_hn3"]

    df = calculate_subscale_score(df, pwb_cols, pwb_reverse, "pwb")
    df = calculate_subscale_score(df, swb_cols, None, "swb")
    df = calculate_subscale_score(df, ewb_cols, ewb_reverse, "ewb")
    df = calculate_subscale_score(df, fwb_cols, None, "fwb")
    df = calculate_subscale_score(df, ecs_cols, ecs_reverse, "ecs")

    # Calculate composite scores with both conditions
    # 1. FACT-G Total (PWB + SWB + EWB + FWB)
    fact_g_items = pwb_cols + swb_cols + ewb_cols + fwb_cols  # 27 items
    fact_g_item_cols = [f"{col}_score" for col in fact_g_items]
    fact_g_num_answered = df[fact_g_item_cols].notna().sum(axis=1)
    fact_g_min_items = 22  # 80% of 27 items = 21.6, rounded up to 22
    
    df['fact_g_total'] = np.where(
        # Condition 1: All subscales must be non-NaN (already ensures 50% of items per subscale)
        (df["pwb_subscale_score"].notna()) & 
        (df["swb_subscale_score"].notna()) & 
        (df["ewb_subscale_score"].notna()) & 
        (df["fwb_subscale_score"].notna()) &
        # Condition 2: At least 80% of FACT-G items must be answered
        (fact_g_num_answered >= fact_g_min_items),
        df["pwb_subscale_score"] + df["swb_subscale_score"] + 
        df["ewb_subscale_score"] + df["fwb_subscale_score"],
        np.nan
    )
    
    # 2. FACT-E Total (FACT-G + ECS)
    fact_e_items = fact_g_items + ecs_cols  # 27 + 17 = 44 items
    fact_e_item_cols = [f"{col}_score" for col in fact_e_items]
    fact_e_num_answered = df[fact_e_item_cols].notna().sum(axis=1)
    fact_e_min_items = 36  # 80% of 44 items = 35.2, rounded up to 36
    
    df['fact_e_total'] = np.where(
        # Condition 1: FACT-G and ECS must be non-NaN
        (df['fact_g_total'].notna()) & 
        (df["ecs_subscale_score"].notna()) &
        # Condition 2: At least 80% of FACT-E items must be answered
        (fact_e_num_answered >= fact_e_min_items),
        df['fact_g_total'] + df["ecs_subscale_score"],
        np.nan
    )
    
    # 3. TOI (PWB + FWB + ECS)
    toi_items = pwb_cols + fwb_cols + ecs_cols  # 7 + 7 + 17 = 31 items
    toi_item_cols = [f"{col}_score" for col in toi_items]
    toi_num_answered = df[toi_item_cols].notna().sum(axis=1)
    toi_min_items = 25  # 80% of 31 items = 24.8, rounded up to 25
    
    df['toi'] = np.where(
        # Condition 1: PWB, FWB, and ECS must be non-NaN
        (df["pwb_subscale_score"].notna()) & 
        (df["fwb_subscale_score"].notna()) & 
        (df["ecs_subscale_score"].notna()) &
        # Condition 2: At least 80% of TOI items must be answered
        (toi_num_answered >= toi_min_items),
        df["pwb_subscale_score"] + df["fwb_subscale_score"] + df["ecs_subscale_score"],
        np.nan
    )

    return df

# Step 4: Analyze trajectory for all scores (fixed)
def analyze_trajectories(df):
    results = []
    unique_ids = df['id'].unique()
    
    score_types = ['pwb_subscale_score', 'swb_subscale_score', 'ewb_subscale_score',
                    'fwb_subscale_score', 'ecs_subscale_score', 'fact_g_total',
                    'fact_e_total', 'toi']
    
    for id_val in unique_ids:
        id_data = df[df['id'] == id_val]
        events = id_data['redcap_event_name'].unique()
        print(f"\nID {id_val} has events: {events}")
        
        event_scores = id_data.groupby('redcap_event_name')[score_types].mean().reset_index()
        
        baseline_score = event_scores[event_scores['redcap_event_name'] == 'baseline_arm_1']
        preop_score = event_scores[event_scores['redcap_event_name'] == 'preoperative_arm_1']
        one_year_score = event_scores[event_scores['redcap_event_name'] == '1_year_postop_arm_1']
        
        # Skip if no one-year data
        if one_year_score.empty:
            continue
            
        # Get values for each timepoint (or NaN if missing)
        one_year_values = one_year_score[score_types].iloc[0].to_dict() if not one_year_score.empty else {k: np.nan for k in score_types}
        
        # Print scores
        if not baseline_score.empty:
            baseline_values = baseline_score[score_types].iloc[0].to_dict()
            print(f"Baseline score: [{', '.join([f'{k}: {v:.1f}' for k, v in baseline_values.items() if pd.notna(v)])}]")
        else:
            print("Baseline score: [nan]")
            
        if not preop_score.empty:
            preop_values = preop_score[score_types].iloc[0].to_dict()
            print(f"Preop score: [{', '.join([f'{k}: {v:.1f}' for k, v in preop_values.items() if pd.notna(v)])}]")
        else:
            print("Preop score: [nan]")
            
        print(f"1-year score: [{', '.join([f'{k}: {v:.1f}' for k, v in one_year_values.items() if pd.notna(v)])}]")

        # Determine which baseline to use (fixed logic)
        baseline_type = None
        baseline_values = None
        
        # Check if baseline_score exists and has non-NaN values
        if not baseline_score.empty:
            baseline_values_temp = baseline_score[score_types].iloc[0].to_dict()
            # Check if at least one score is non-NaN
            if any(pd.notna(val) for val in baseline_values_temp.values()):
                baseline_type = 'baseline_arm_1'
                baseline_values = baseline_values_temp
                print(f"Using baseline score: {', '.join([f'{k}: {v:.1f}' for k, v in baseline_values.items() if pd.notna(v)])}")
        
        # If baseline is not valid, fall back to preop
        if baseline_values is None and not preop_score.empty:
            preop_values_temp = preop_score[score_types].iloc[0].to_dict()
            if any(pd.notna(val) for val in preop_values_temp.values()):
                baseline_type = 'preoperative_arm_1'
                baseline_values = preop_values_temp
                print(f"Using preop score as baseline: {', '.join([f'{k}: {v:.1f}' for k, v in baseline_values.items() if pd.notna(v)])}")
        
        # Skip if no valid baseline
        if baseline_values is None:
            continue
        
        # Calculate improvements
        result = {'id': id_val, 'baseline_type': baseline_type}
        for score_type in score_types:
            result.update({
                f'{score_type}_baseline': baseline_values[score_type],
                f'{score_type}_one_year': one_year_values[score_type]
            })
        
        results.append(result)
    
    return pd.DataFrame(results)

# Main function (unchanged)
def analyze_patient_outcomes(df):
    required_cols = ['id', 'redcap_event_name'] + \
                    [f"gp{i}" for i in range(1, 8)] + \
                    [f"gs{i}" for i in range(1, 8)] + \
                    [f"ge{i}" for i in range(1, 7)] + \
                    [f"gf{i}" for i in range(1, 8)] + \
                    [f"a_hn{i}" for i in range(1, 6)] + ["a_hn7", "a_hn10"] + \
                    [f"a_e{i}" for i in range(1, 8)] + ["a_c6", "a_c2", "a_act11"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"Warning: Missing columns: {missing_cols}")
    
    eligible_ids = find_eligible_ids(df)
    print(f"Found {len(eligible_ids)} eligible IDs")
    
    filtered_df = filter_eligible_data(df, eligible_ids)
    print(f"Filtered dataframe shape: {filtered_df.shape}")
    
    filtered_df = calculate_all_scores(filtered_df)
    
    score_cols = [col for col in filtered_df.columns if col.endswith('_score') or col.endswith('_total')]
    print(f"Created score columns: {score_cols}")
    
    score_types = ['pwb_subscale_score', 'swb_subscale_score', 'ewb_subscale_score',
                    'fwb_subscale_score', 'ecs_subscale_score', 'fact_g_total',
                    'fact_e_total', 'toi']
    for score_type in score_types:
        non_nan_count = filtered_df[score_type].notna().sum()
        print(f"Non-NaN {score_type}: {non_nan_count} out of {len(filtered_df)}")
    
    results_df = analyze_trajectories(filtered_df)
    
    if not results_df.empty:
        print(f"\nAnalysis complete for {len(results_df)} patients:")
        for score_type in score_types:
            # Count patients with valid baseline scores for this score type
            valid_baselines = results_df[f'{score_type}_baseline'].notna().sum()
            print(f"Patients with valid {score_type} baseline: {valid_baselines} out of {len(results_df)}")
    return filtered_df, results_df

# Example usage
filtered_df, results_df = analyze_patient_outcomes(df)

results_df.to_csv("results.csv", index=False)

Found 167 eligible IDs
Filtered dataframe shape: (2235, 84)
Created score columns: ['cci_total', 'gp1_score', 'gp2_score', 'gp3_score', 'gp4_score', 'gp5_score', 'gp6_score', 'gp7_score', 'pwb_subscale_score', 'gs1_score', 'gs2_score', 'gs3_score', 'gs4_score', 'gs5_score', 'gs6_score', 'gs7_score', 'swb_subscale_score', 'ge1_score', 'ge2_score', 'ge3_score', 'ge4_score', 'ge5_score', 'ge6_score', 'ewb_subscale_score', 'gf1_score', 'gf2_score', 'gf3_score', 'gf4_score', 'gf5_score', 'gf6_score', 'gf7_score', 'fwb_subscale_score', 'a_hn1_score', 'a_hn2_score', 'a_hn3_score', 'a_hn4_score', 'a_hn5_score', 'a_hn7_score', 'a_hn10_score', 'a_e1_score', 'a_e2_score', 'a_e3_score', 'a_e4_score', 'a_e5_score', 'a_e6_score', 'a_e7_score', 'a_c6_score', 'a_c2_score', 'a_act11_score', 'ecs_subscale_score', 'fact_g_total', 'fact_e_total']
Non-NaN pwb_subscale_score: 622 out of 2235
Non-NaN swb_subscale_score: 615 out of 2235
Non-NaN ewb_subscale_score: 621 out of 2235
Non-NaN fwb_subscale_score: 6

In [8]:
results_df

,id,baseline_type,pwb_subscale_score_baseline,pwb_subscale_score_one_year,swb_subscale_score_baseline,swb_subscale_score_one_year,ewb_subscale_score_baseline,ewb_subscale_score_one_year,fwb_subscale_score_baseline,fwb_subscale_score_one_year,ecs_subscale_score_baseline,ecs_subscale_score_one_year,fact_g_total_baseline,fact_g_total_one_year,fact_e_total_baseline,fact_e_total_one_year,toi_baseline,toi_one_year
0,32,baseline_arm_1,24.5,27.0,22.000000,28.0,6.0,24.0,19.0,25.0,46.0,55.0,71.500000,104.0,117.5,159.0,89.5,107.0
1,76,preoperative_arm_1,7.0,7.0,23.000000,21.0,14.0,11.0,15.0,10.0,36.0,48.0,59.000000,49.0,95.0,97.0,58.0,65.0
2,125,baseline_arm_1,21.0,22.0,17.500000,21.0,21.0,19.0,8.0,13.0,52.0,54.0,67.500000,75.0,119.5,129.0,81.0,89.0
3,160,baseline_arm_1,21.0,28.0,20.000000,22.0,21.0,23.0,20.0,22.0,37.0,57.0,82.000000,95.0,119.0,152.0,78.0,107.0
4,198,baseline_arm_1,25.0,26.5,13.000000,18.0,17.0,23.0,21.0,20.5,57.0,51.0,76.000000,88.0,133.0,139.0,103.0,98.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,1571,baseline_arm_1,26.0,16.0,28.000000,28.0,16.0,15.0,25.0,10.0,58.0,35.0,95.000000,69.0,153.0,104.0,109.0,61.0
120,1572,baseline_arm_1,27.0,24.0,13.000000,21.0,19.0,18.0,20.0,15.0,62.0,37.0,79.000000,78.0,141.0,115.0,109.0,76.0
121,1615,baseline_arm_1,23.0,19.0,24.000000,22.0,16.0,20.0,12.0,13.0,51.0,37.0,75.000000,74.0,126.0,111.0,86.0,69.0
122,1653,baseline_arm_1,21.0,26.0,19.000000,21.0,18.0,16.0,24.0,28.0,30.0,62.0,82.000000,91.0,112.0,153.0,75.0,116.0


In [9]:
filtered_df

,id,operation_date,redcap_event_name,qol_date,age_diagnosis,dob,overall_primary_tumour,overall_regional_ln,overall_distant_metastasis,neotx___notx,...,a_e5_score,a_e6_score,a_e7_score,a_c6_score,a_c2_score,a_act11_score,ecs_subscale_score,fact_g_total,fact_e_total,toi
235,32,NaT,baseline_arm_1,NaT,NaN,1977-09-24,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
236,32,2007-11-15,surgery_arm_1,NaT,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
237,32,NaT,1_month_postop_arm_1,NaT,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
238,32,NaT,surgery_arm_1,NaT,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
239,32,NaT,baseline_arm_1,NaT,NaN,NaT,NaN,NaN,NaN,NaN,...,4.0,2.0,4.0,3.0,2.0,2.0,46.0000,71.500000,117.500000,89.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11537,1690,NaT,preoperative_arm_1,NaT,NaN,NaT,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11538,1690,NaT,1_year_postop_arm_1,NaT,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11539,1690,NaT,baseline_arm_1,NaT,NaN,1977-08-25,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11540,1690,NaT,surgery_arm_1,NaT,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
filtered_df['id'].nunique()

167

In [11]:
results_df['id'].nunique()

124

# Data set up

In [12]:
results_df = pd.read_csv("results.csv")

In [13]:
results_df

,id,baseline_type,pwb_subscale_score_baseline,pwb_subscale_score_one_year,swb_subscale_score_baseline,swb_subscale_score_one_year,ewb_subscale_score_baseline,ewb_subscale_score_one_year,fwb_subscale_score_baseline,fwb_subscale_score_one_year,ecs_subscale_score_baseline,ecs_subscale_score_one_year,fact_g_total_baseline,fact_g_total_one_year,fact_e_total_baseline,fact_e_total_one_year,toi_baseline,toi_one_year
0,32,baseline_arm_1,24.5,27.0,22.000000,28.0,6.0,24.0,19.0,25.0,46.0,55.0,71.500000,104.0,117.5,159.0,89.5,107.0
1,76,preoperative_arm_1,7.0,7.0,23.000000,21.0,14.0,11.0,15.0,10.0,36.0,48.0,59.000000,49.0,95.0,97.0,58.0,65.0
2,125,baseline_arm_1,21.0,22.0,17.500000,21.0,21.0,19.0,8.0,13.0,52.0,54.0,67.500000,75.0,119.5,129.0,81.0,89.0
3,160,baseline_arm_1,21.0,28.0,20.000000,22.0,21.0,23.0,20.0,22.0,37.0,57.0,82.000000,95.0,119.0,152.0,78.0,107.0
4,198,baseline_arm_1,25.0,26.5,13.000000,18.0,17.0,23.0,21.0,20.5,57.0,51.0,76.000000,88.0,133.0,139.0,103.0,98.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,1571,baseline_arm_1,26.0,16.0,28.000000,28.0,16.0,15.0,25.0,10.0,58.0,35.0,95.000000,69.0,153.0,104.0,109.0,61.0
120,1572,baseline_arm_1,27.0,24.0,13.000000,21.0,19.0,18.0,20.0,15.0,62.0,37.0,79.000000,78.0,141.0,115.0,109.0,76.0
121,1615,baseline_arm_1,23.0,19.0,24.000000,22.0,16.0,20.0,12.0,13.0,51.0,37.0,75.000000,74.0,126.0,111.0,86.0,69.0
122,1653,baseline_arm_1,21.0,26.0,19.000000,21.0,18.0,16.0,24.0,28.0,30.0,62.0,82.000000,91.0,112.0,153.0,75.0,116.0


In [14]:
# Load results_df before the filtering
results_df_original = pd.read_csv("results.csv")

# Select relevant columns for FACT-E (baseline and 12-month only)
results_df_facte = results_df_original[['id', 'baseline_type', 'fact_e_total_baseline', 'fact_e_total_one_year']]

# Rename columns
results_df_facte = results_df_facte.rename(columns={
    "fact_e_total_baseline": "Baseline",
    "fact_e_total_one_year": "12mo"
})

print(f"Total patients in results_df: {len(results_df_facte)}")

# Check baseline FACT-E availability
has_baseline = results_df_facte['Baseline'].notna().sum()
print(f"\nPatients with valid baseline FACT-E: {has_baseline}")

# Check 12-month availability
has_12mo = results_df_facte['12mo'].notna().sum()
print(f"Patients with 12-month FACT-E: {has_12mo}")

# Baseline but NO 12-month
has_baseline_only = (results_df_facte['Baseline'].notna() & 
                     results_df_facte['12mo'].isna()).sum()
print(f"\nPatients with baseline FACT-E but NO 12-month: {has_baseline_only}")

# 12-month but NO baseline
has_12mo_only = (results_df_facte['Baseline'].isna() & 
                 results_df_facte['12mo'].notna()).sum()
print(f"Patients with 12-month FACT-E but NO baseline: {has_12mo_only}")

# Both baseline AND 12-month
has_both = (results_df_facte['Baseline'].notna() & 
            results_df_facte['12mo'].notna()).sum()
print(f"Patients with baseline FACT-E AND 12-month: {has_both}")

Total patients in results_df: 124

Patients with valid baseline FACT-E: 116
Patients with 12-month FACT-E: 114

Patients with baseline FACT-E but NO 12-month: 8
Patients with 12-month FACT-E but NO baseline: 6
Patients with baseline FACT-E AND 12-month: 108


In [15]:
# Select relevant columns for baseline and 12-month analysis
results_df = results_df[['id', 'baseline_type', 'fact_e_total_baseline', 'fact_e_total_one_year']]

# Rename columns to match timepoints
results_df = results_df.rename(columns={
    "fact_e_total_baseline": "Baseline",
    "fact_e_total_one_year": "12mo"
})

# Filter to patients who have baseline AND 12-month follow-up
has_baseline = results_df["Baseline"].notna()
has_12mo = results_df["12mo"].notna()
results_df = results_df[has_baseline & has_12mo]

print(f"Patients with valid baseline and 12-month follow-up: {len(results_df)}")


Patients with valid baseline and 12-month follow-up: 108


In [16]:
# Set 'id' column as the index
results_df = results_df.set_index("id")

# Rename the index and columns
results_df.index.name = "id"
results_df.columns.name = "Timepoint"

results_df

score_comparison = results_df.copy()
score_comparison

Timepoint,baseline_type,Baseline,12mo
id,,,
32,baseline_arm_1,117.5,159.0
76,preoperative_arm_1,95.0,97.0
125,baseline_arm_1,119.5,129.0
160,baseline_arm_1,119.0,152.0
198,baseline_arm_1,133.0,139.0
...,...,...,...
1566,preoperative_arm_1,150.0,107.0
1571,baseline_arm_1,153.0,104.0
1572,baseline_arm_1,141.0,115.0


In [17]:
score_comparison['baseline_type'].value_counts()

baseline_type
baseline_arm_1        96
preoperative_arm_1    12
Name: count, dtype: int64

In [18]:
# Start with the 108 patients from score_comparison (FACT-E cohort)
eligible_patient_ids = score_comparison.index.tolist()

# Load the original results_df
results_df_original = pd.read_csv("results.csv")

# Filter to only the 108 eligible patients
results_df_filtered = results_df_original[results_df_original['id'].isin(eligible_patient_ids)].copy()

# Define all score types
score_types = ['fact_e_total', 'pwb_subscale_score', 'swb_subscale_score', 
               'ewb_subscale_score', 'fwb_subscale_score', 'ecs_subscale_score']

# Select relevant columns
cols_to_keep = ['id', 'baseline_type']
for score_type in score_types:
    cols_to_keep.extend([f'{score_type}_baseline', f'{score_type}_one_year'])

# Create extended dataframe
score_comparison_extended = results_df_filtered[cols_to_keep].copy()

# Rename columns to use consistent naming
rename_dict = {'baseline_type': 'baseline_type'}
for score_type in score_types:
    rename_dict[f'{score_type}_baseline'] = f'{score_type}_Baseline'
    rename_dict[f'{score_type}_one_year'] = f'{score_type}_12mo'

score_comparison_extended = score_comparison_extended.rename(columns=rename_dict)

# Set id as index
score_comparison_extended = score_comparison_extended.set_index('id')
score_comparison_extended.index.name = 'id'

# Display summary
print(f"Extended score_comparison shape: {score_comparison_extended.shape}")
print(f"Number of patients: {len(score_comparison_extended)}")
print(f"\nColumns: {list(score_comparison_extended.columns)}")

# Check completeness for each domain
print(f"\n{'='*80}")
print("DATA COMPLETENESS BY DOMAIN (Baseline + 12mo)")
print(f"{'='*80}")
for score_type in score_types:
    baseline_col = f'{score_type}_Baseline'
    mo12_col = f'{score_type}_12mo'
    
    n_baseline = score_comparison_extended[baseline_col].notna().sum()
    n_12mo = score_comparison_extended[mo12_col].notna().sum()
    n_both = (score_comparison_extended[baseline_col].notna() & 
              score_comparison_extended[mo12_col].notna()).sum()
    
    print(f"{score_type:25} Baseline: {n_baseline:3}/108  |  12mo: {n_12mo:3}/108  |  Both: {n_both:3}/108")

# Update score_comparison to the extended version
score_comparison = score_comparison_extended.copy()
print(f"\n{'='*80}")
print(f"Updated 'score_comparison' with all 6 domains for {len(score_comparison)} patients")
print(f"{'='*80}")

Extended score_comparison shape: (108, 13)
Number of patients: 108

Columns: ['baseline_type', 'fact_e_total_Baseline', 'fact_e_total_12mo', 'pwb_subscale_score_Baseline', 'pwb_subscale_score_12mo', 'swb_subscale_score_Baseline', 'swb_subscale_score_12mo', 'ewb_subscale_score_Baseline', 'ewb_subscale_score_12mo', 'fwb_subscale_score_Baseline', 'fwb_subscale_score_12mo', 'ecs_subscale_score_Baseline', 'ecs_subscale_score_12mo']

DATA COMPLETENESS BY DOMAIN (Baseline + 12mo)
fact_e_total              Baseline: 108/108  |  12mo: 108/108  |  Both: 108/108
pwb_subscale_score        Baseline: 108/108  |  12mo: 108/108  |  Both: 108/108
swb_subscale_score        Baseline: 108/108  |  12mo: 108/108  |  Both: 108/108
ewb_subscale_score        Baseline: 108/108  |  12mo: 108/108  |  Both: 108/108
fwb_subscale_score        Baseline: 108/108  |  12mo: 108/108  |  Both: 108/108
ecs_subscale_score        Baseline: 108/108  |  12mo: 108/108  |  Both: 108/108

Updated 'score_comparison' with all 6 dom

In [19]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,fwb_subscale_score_12mo,ecs_subscale_score_Baseline,ecs_subscale_score_12mo
id,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,25.0,46.0,55.0
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,10.0,36.0,48.0
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,13.0,52.0,54.0
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,22.0,37.0,57.0
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,20.5,57.0,51.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,16.0,67.0,38.0
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,10.0,58.0,35.0
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,15.0,62.0,37.0


# Characteristic by class - 0) FACT E baseline 

In [90]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,fwb_subscale_score_12mo,ecs_subscale_score_Baseline,ecs_subscale_score_12mo
id,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,25.0,46.0,55.0
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,10.0,36.0,48.0
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,13.0,52.0,54.0
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,22.0,37.0,57.0
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,20.5,57.0,51.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,16.0,67.0,38.0
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,10.0,58.0,35.0
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,15.0,62.0,37.0


In [91]:
def preprocess_facte_baseline(score_comparison):
    """
    Preprocess baseline FACT-E score: z-score standardization (no imputation needed as no missing values)
    
    Returns:
    - Dictionary with preprocessing results and statistics
    """
    # Check for missing values
    n_missing = score_comparison['fact_e_total_Baseline'].isna().sum()
    n_total = len(score_comparison)
    
    print("="*80)
    print("STEP 2: PREDICTOR PREPROCESSING - FACT-E Baseline")
    print("="*80)
    print(f"Total patients: {n_total}")
    print(f"Missing FACT-E baseline: {n_missing}")
    print(f"Complete FACT-E baseline: {n_total - n_missing}")
    
    if n_missing > 0:
        print(f"\nWARNING: {n_missing} patients have missing baseline FACT-E")
    else:
        print(f"\n✓ No missing values - proceeding with z-score standardization")
    
    # Calculate mean and SD
    mean_facte = score_comparison['fact_e_total_Baseline'].mean()
    sd_facte = score_comparison['fact_e_total_Baseline'].std()
    
    print(f"\nBaseline FACT-E statistics:")
    print(f"  Mean: {mean_facte:.2f}")
    print(f"  SD: {sd_facte:.2f}")
    print(f"  Min: {score_comparison['fact_e_total_Baseline'].min():.2f}")
    print(f"  Max: {score_comparison['fact_e_total_Baseline'].max():.2f}")
    
    # Z-score standardization
    score_comparison['fact_e_total_Baseline_std'] = (
        (score_comparison['fact_e_total_Baseline'] - mean_facte) / sd_facte
    )
    
    print(f"\nStandardized FACT-E statistics:")
    print(f"  Mean: {score_comparison['fact_e_total_Baseline_std'].mean():.2f}")
    print(f"  SD: {score_comparison['fact_e_total_Baseline_std'].std():.2f}")
    print(f"  Min: {score_comparison['fact_e_total_Baseline_std'].min():.2f}")
    print(f"  Max: {score_comparison['fact_e_total_Baseline_std'].max():.2f}")
    
    print("\n✓ Created column: 'fact_e_total_Baseline_std'")
    print("="*80)
    
    # Return results dictionary
    results = {
        'n_total': n_total,
        'n_missing': n_missing,
        'n_complete': n_total - n_missing,
        'mean': mean_facte,
        'sd': sd_facte,
        'min': score_comparison['fact_e_total_Baseline'].min(),
        'max': score_comparison['fact_e_total_Baseline'].max(),
        'preprocessed_data': score_comparison
    }
    
    return results

# Run preprocessing
baseline_summaries = preprocess_facte_baseline(score_comparison)

# Update score_comparison with preprocessed data
score_comparison = baseline_summaries['preprocessed_data']

score_comparison[['fact_e_total_Baseline', 'fact_e_total_Baseline_std']]

STEP 2: PREDICTOR PREPROCESSING - FACT-E Baseline
Total patients: 108
Missing FACT-E baseline: 0
Complete FACT-E baseline: 108

✓ No missing values - proceeding with z-score standardization

Baseline FACT-E statistics:
  Mean: 128.84
  SD: 23.35
  Min: 69.07
  Max: 176.00

Standardized FACT-E statistics:
  Mean: -0.00
  SD: 1.00
  Min: -2.56
  Max: 2.02

✓ Created column: 'fact_e_total_Baseline_std'


,fact_e_total_Baseline,fact_e_total_Baseline_std
id,,
32,117.5,-0.485421
76,95.0,-1.448897
125,119.5,-0.399778
160,119.0,-0.421189
198,133.0,0.178308
...,...,...
1566,150.0,0.906268
1571,153.0,1.034731
1572,141.0,0.520877


In [92]:
baseline_summaries

{'n_total': 108,
 'n_missing': 0,
 'n_complete': 108,
 'mean': 128.83599653946877,
 'sd': 23.352931602311035,
 'min': 69.07083333333333,
 'max': 176.0,
 'preprocessed_data':            baseline_type  fact_e_total_Baseline  fact_e_total_12mo  \
 id                                                                   
 32        baseline_arm_1                  117.5              159.0   
 76    preoperative_arm_1                   95.0               97.0   
 125       baseline_arm_1                  119.5              129.0   
 160       baseline_arm_1                  119.0              152.0   
 198       baseline_arm_1                  133.0              139.0   
 ...                  ...                    ...                ...   
 1566  preoperative_arm_1                  150.0              107.0   
 1571      baseline_arm_1                  153.0              104.0   
 1572      baseline_arm_1                  141.0              115.0   
 1615      baseline_arm_1                  126

In [93]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,fwb_subscale_score_12mo,ecs_subscale_score_Baseline,ecs_subscale_score_12mo,fact_e_total_Baseline_std
id,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,25.0,46.0,55.0,-0.485421
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,10.0,36.0,48.0,-1.448897
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,13.0,52.0,54.0,-0.399778
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,22.0,37.0,57.0,-0.421189
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,20.5,57.0,51.0,0.178308
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,16.0,67.0,38.0,0.906268
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,10.0,58.0,35.0,1.034731
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,15.0,62.0,37.0,0.520877


In [94]:
baseline_summaries

{'n_total': 108,
 'n_missing': 0,
 'n_complete': 108,
 'mean': 128.83599653946877,
 'sd': 23.352931602311035,
 'min': 69.07083333333333,
 'max': 176.0,
 'preprocessed_data':            baseline_type  fact_e_total_Baseline  fact_e_total_12mo  \
 id                                                                   
 32        baseline_arm_1                  117.5              159.0   
 76    preoperative_arm_1                   95.0               97.0   
 125       baseline_arm_1                  119.5              129.0   
 160       baseline_arm_1                  119.0              152.0   
 198       baseline_arm_1                  133.0              139.0   
 ...                  ...                    ...                ...   
 1566  preoperative_arm_1                  150.0              107.0   
 1571      baseline_arm_1                  153.0              104.0   
 1572      baseline_arm_1                  141.0              115.0   
 1615      baseline_arm_1                  126

# Characteristic by class - 1) BMI 

In [95]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,fwb_subscale_score_12mo,ecs_subscale_score_Baseline,ecs_subscale_score_12mo,fact_e_total_Baseline_std
id,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,25.0,46.0,55.0,-0.485421
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,10.0,36.0,48.0,-1.448897
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,13.0,52.0,54.0,-0.399778
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,22.0,37.0,57.0,-0.421189
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,20.5,57.0,51.0,0.178308
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,16.0,67.0,38.0,0.906268
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,10.0,58.0,35.0,1.034731
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,15.0,62.0,37.0,0.520877


In [96]:
def preprocess_bmi(score_comparison, filtered_df):
    """
    Preprocess BMI: median imputation for missing/outliers + z-score standardization
    
    Returns:
    - Dictionary with preprocessing results and statistics
    """
    
    print("="*80)
    print("STEP 2: PREDICTOR PREPROCESSING - BMI")
    print("="*80)
    
    # Get eligible patient IDs from score_comparison
    eligible_patient_ids = score_comparison.index.tolist()
    
    # Create dataframe with patient IDs
    bmi_prep = pd.DataFrame({'patient_id': eligible_patient_ids})
    
    # Merge with filtered_df to get BMI
    bmi_prep = bmi_prep.merge(
        filtered_df[["id", "bmi"]],
        left_on='patient_id',
        right_on='id',
        how='left'
    )
    
    print(f"Total patients: {len(eligible_patient_ids)}")
    
    # Calculate median BMI from all non-missing values
    median_bmi = bmi_prep['bmi'].median()
    print(f"\nMedian BMI (for imputation): {median_bmi:.2f}")
    
    # Find patient IDs where all BMI values are missing
    patients_all_missing = bmi_prep.groupby('patient_id')['bmi'].apply(lambda x: x.isna().all())
    patients_to_impute = patients_all_missing[patients_all_missing].index
    
    # Impute BMI for those patients
    mask = bmi_prep['patient_id'].isin(patients_to_impute)
    bmi_prep.loc[mask, 'bmi'] = median_bmi
    print(f"Imputed {len(patients_to_impute)} missing patients with median BMI: {median_bmi:.2f}")
    
    # Replace BMI outliers with median (outside range 7-186)
    outlier_mask = (bmi_prep['bmi'] < 7) | (bmi_prep['bmi'] > 186)
    outlier_unique_patients = bmi_prep.loc[outlier_mask, 'patient_id'].nunique()
    outlier_count = outlier_mask.sum()
    bmi_prep.loc[outlier_mask, 'bmi'] = median_bmi
    print(f"Replaced {outlier_count} outlier BMI values from {outlier_unique_patients} unique patients with median BMI: {median_bmi:.2f}")
    
    # Remove NA and duplicates (critical because filtered_df has multiple rows per patient)
    bmi_prep_clean = bmi_prep.dropna(subset=["bmi"])
    bmi_prep_clean = bmi_prep_clean.drop_duplicates()
    
    # Calculate mean and SD for z-score standardization
    mean_bmi = bmi_prep_clean['bmi'].mean()
    sd_bmi = bmi_prep_clean['bmi'].std()
    
    print(f"\nBMI statistics:")
    print(f"  N: {len(bmi_prep_clean)}")
    print(f"  Mean: {mean_bmi:.2f}")
    print(f"  SD: {sd_bmi:.2f}")
    print(f"  Min: {bmi_prep_clean['bmi'].min():.2f}")
    print(f"  Max: {bmi_prep_clean['bmi'].max():.2f}")
    
    # Z-score standardization
    bmi_prep_clean['bmi_std'] = (bmi_prep_clean['bmi'] - mean_bmi) / sd_bmi
    
    print(f"\nStandardized BMI statistics:")
    print(f"  Mean: {bmi_prep_clean['bmi_std'].mean():.2f}")
    print(f"  SD: {bmi_prep_clean['bmi_std'].std():.2f}")
    print(f"  Min: {bmi_prep_clean['bmi_std'].min():.2f}")
    print(f"  Max: {bmi_prep_clean['bmi_std'].max():.2f}")
    
    # Merge back to score_comparison
    bmi_final = bmi_prep_clean[['patient_id', 'bmi', 'bmi_std']].drop_duplicates(subset=['patient_id'])
    bmi_final = bmi_final.set_index('patient_id')
    
    score_comparison = score_comparison.merge(
        bmi_final[['bmi', 'bmi_std']], 
        left_index=True, 
        right_index=True, 
        how='left'
    )
    
    print("\n✓ Created columns: 'bmi' and 'bmi_std'")
    print("="*80)
    
    # Return results dictionary
    results = {
        'n_total': len(eligible_patient_ids),
        'n_missing': len(patients_to_impute),
        'n_outliers': outlier_unique_patients,
        'median': median_bmi,
        'mean': mean_bmi,
        'sd': sd_bmi,
        'preprocessed_data': score_comparison
    }
    
    return results

# Run preprocessing
bmi_summaries = preprocess_bmi(score_comparison, filtered_df)

# Update score_comparison with preprocessed data
score_comparison = bmi_summaries['preprocessed_data']

# Display first few rows to verify
print("\nFirst few rows of BMI (original and standardized):")
print(score_comparison[['bmi', 'bmi_std']].head(10))

STEP 2: PREDICTOR PREPROCESSING - BMI
Total patients: 108

Median BMI (for imputation): 25.86
Imputed 4 missing patients with median BMI: 25.86
Replaced 2 outlier BMI values from 2 unique patients with median BMI: 25.86

BMI statistics:
  N: 108
  Mean: 26.69
  SD: 5.87
  Min: 14.68
  Max: 46.08

Standardized BMI statistics:
  Mean: 0.00
  SD: 1.00
  Min: -2.04
  Max: 3.30

✓ Created columns: 'bmi' and 'bmi_std'

First few rows of BMI (original and standardized):
           bmi   bmi_std
id                      
32   46.075514  3.300320
76   25.360019 -0.226055
125  34.213824  1.281119
160  27.115631  0.072801
198  32.922429  1.061286
230  23.184915 -0.596320
275  22.794163 -0.662837
307  25.856286 -0.141576
364  27.076109  0.066073
442  19.100092 -1.291675


In [97]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,fwb_subscale_score_12mo,ecs_subscale_score_Baseline,ecs_subscale_score_12mo,fact_e_total_Baseline_std,bmi,bmi_std
id,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,25.0,46.0,55.0,-0.485421,46.075514,3.300320
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,10.0,36.0,48.0,-1.448897,25.360019,-0.226055
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,13.0,52.0,54.0,-0.399778,34.213824,1.281119
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,22.0,37.0,57.0,-0.421189,27.115631,0.072801
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,20.5,57.0,51.0,0.178308,32.922429,1.061286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,16.0,67.0,38.0,0.906268,27.297959,0.103839
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,10.0,58.0,35.0,1.034731,33.217993,1.111599
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,15.0,62.0,37.0,0.520877,39.100346,2.112946


In [98]:
bmi_summaries

{'n_total': 108,
 'n_missing': 4,
 'n_outliers': 2,
 'median': 25.85628628,
 'mean': 26.687964146759253,
 'sd': 5.874444620540978,
 'preprocessed_data':            baseline_type  fact_e_total_Baseline  fact_e_total_12mo  \
 id                                                                   
 32        baseline_arm_1                  117.5              159.0   
 76    preoperative_arm_1                   95.0               97.0   
 125       baseline_arm_1                  119.5              129.0   
 160       baseline_arm_1                  119.0              152.0   
 198       baseline_arm_1                  133.0              139.0   
 ...                  ...                    ...                ...   
 1566  preoperative_arm_1                  150.0              107.0   
 1571      baseline_arm_1                  153.0              104.0   
 1572      baseline_arm_1                  141.0              115.0   
 1615      baseline_arm_1                  126.0              111.0

# Characteristic by class - 2) smoking

In [99]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,fwb_subscale_score_12mo,ecs_subscale_score_Baseline,ecs_subscale_score_12mo,fact_e_total_Baseline_std,bmi,bmi_std
id,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,25.0,46.0,55.0,-0.485421,46.075514,3.300320
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,10.0,36.0,48.0,-1.448897,25.360019,-0.226055
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,13.0,52.0,54.0,-0.399778,34.213824,1.281119
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,22.0,37.0,57.0,-0.421189,27.115631,0.072801
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,20.5,57.0,51.0,0.178308,32.922429,1.061286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,16.0,67.0,38.0,0.906268,27.297959,0.103839
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,10.0,58.0,35.0,1.034731,33.217993,1.111599
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,15.0,62.0,37.0,0.520877,39.100346,2.112946


In [100]:
def preprocess_smoking(score_comparison, filtered_df):
    """
    Preprocess smoking: collapse categories + MICE imputation for missing
    Smoking is categorical: 0=Never, 1=Ever (collapsed from 1,2,3), 9999=Unknown (then imputed with MICE)
    
    Returns:
    - Dictionary with preprocessing results and statistics
    """
    import miceforest as mf
    
    print("="*80)
    print("STEP 2: PREDICTOR PREPROCESSING - Smoking")
    print("="*80)
    print("Preparing smoking data for MICE imputation...")
    
    # Get all unique patient IDs from score_comparison
    all_patient_ids = score_comparison.index.tolist()
    
    # Merge with filtered_df to get smoking
    smoking_prep = pd.DataFrame({'patient_id': all_patient_ids})
    smoking_prep = smoking_prep.merge(
        filtered_df[["id", "smoking"]],
        left_on='patient_id',
        right_on='id',
        how='left'
    )
    
    print(smoking_prep['smoking'].value_counts())
    
    # Find patient IDs where all smoking values are missing
    patients_all_missing = smoking_prep.groupby('patient_id')['smoking'].apply(lambda x: x.isna().all())
    patients_to_impute = patients_all_missing[patients_all_missing].index
    
    # Impute smoking for those patients with 9999 (unknown)
    mask = smoking_prep['patient_id'].isin(patients_to_impute)
    smoking_prep.loc[mask, 'smoking'] = 9999.0
    print(f"Imputed {len(patients_to_impute)} unique patients with smoking=9999 (unknown)")
    
    # Remove NA and duplicates
    smoking_prep_clean = smoking_prep.dropna(subset=["smoking"])
    smoking_prep_clean = smoking_prep_clean.drop_duplicates()
    
    print(smoking_prep_clean['smoking'].value_counts())
    
    # Collapse smoking variable
    print("\nCollapsing smoking variable...")
    smoking_prep_clean['smoking_collapsed'] = smoking_prep_clean['smoking'].copy()
    smoking_prep_clean.loc[smoking_prep_clean['smoking'].isin([1, 2, 3]), 'smoking_collapsed'] = 1  # Ever Smoked
    smoking_prep_clean.loc[smoking_prep_clean['smoking'] == 0, 'smoking_collapsed'] = 0  # Never Smoked
    smoking_prep_clean.loc[smoking_prep_clean['smoking'] == 9999, 'smoking_collapsed'] = 9999  # Unknown
    
    print("Smoking collapsed - Value counts:")
    print(smoking_prep_clean['smoking_collapsed'].value_counts())
    
    # Perform MICE imputation ONCE
    print("\nPerforming MICE imputation for smoking=9999...")
    
    # Replace 9999 with NaN for imputation
    smoking_prep_clean['smoking_collapsed_for_imputation'] = smoking_prep_clean['smoking_collapsed'].replace(9999, np.nan)
    
    # Merge baseline FACT-E for imputation model
    smoking_prep_clean = smoking_prep_clean.merge(
        score_comparison[['fact_e_total_Baseline']],
        left_on='patient_id',
        right_index=True,
        how='left'
    )
    
    print(f"Running MICE on {len(smoking_prep_clean)} unique patients")
    
    # Create dataset for MICE
    mice_data = smoking_prep_clean[['smoking_collapsed_for_imputation', 'fact_e_total_Baseline']].copy()
    
    # MICE imputation
    mice_imputer = mf.ImputationKernel(
        mice_data,
        datasets=3,
        random_state=42
    )
    
    mice_imputer.mice(iterations=3, verbose=True)
    
    # Get completed dataset
    completed_data = mice_imputer.complete_data(0)
    
    # Extract imputed values
    imputed_smoking = completed_data['smoking_collapsed_for_imputation'].round().astype(int)
    
    # Create mapping dictionary for all patients
    smoking_imputation_map = dict(zip(smoking_prep_clean['patient_id'], imputed_smoking))
    
    print("Smoking after MICE imputation - Value counts:")
    print(pd.Series(imputed_smoking).value_counts())
    print("\n" + "="*80)
    
    # Create final dataframe with imputed smoking values
    smoking_final = pd.DataFrame({
        'patient_id': list(smoking_imputation_map.keys()),
        'smoking_collapsed_imputed': list(smoking_imputation_map.values())
    })
    smoking_final = smoking_final.set_index('patient_id')
    
    # Calculate summary statistics
    smoking_data = smoking_final['smoking_collapsed_imputed']
    n_never = (smoking_data == 0).sum()
    n_ever = (smoking_data == 1).sum()
    total = len(smoking_data)
    
    print(f"\nFinal smoking distribution:")
    print(f"  N: {total}")
    print(f"  Never (0): {n_never} ({n_never/total*100:.1f}%)")
    print(f"  Ever (1): {n_ever} ({n_ever/total*100:.1f}%)")
    print(f"  Mean: {smoking_data.mean():.2f}")
    print(f"  SD: {smoking_data.std():.2f}")
    
    # Merge back to score_comparison
    score_comparison = score_comparison.merge(
        smoking_final[['smoking_collapsed_imputed']], 
        left_index=True, 
        right_index=True, 
        how='left'
    )
    
    print("\n✓ Created column: 'smoking_collapsed_imputed'")
    print("="*80)
    
    # Return results dictionary
    results = {
        'n_total': len(all_patient_ids),
        'n_missing': len(patients_to_impute),
        'n_never': n_never,
        'n_ever': n_ever,
        'mean': smoking_data.mean(),
        'sd': smoking_data.std(),
        'preprocessed_data': score_comparison
    }
    
    return results

# Run preprocessing
smoking_summaries = preprocess_smoking(score_comparison, filtered_df)

# Update score_comparison with preprocessed data
score_comparison = smoking_summaries['preprocessed_data']

# Display first few rows to verify
print("\nFirst few rows of smoking (imputed):")
print(score_comparison[['smoking_collapsed_imputed']].head(10))

STEP 2: PREDICTOR PREPROCESSING - Smoking
Preparing smoking data for MICE imputation...
smoking
3.0       31
0.0       29
1.0       16
9999.0     6
Name: count, dtype: int64
Imputed 26 unique patients with smoking=9999 (unknown)
smoking
9999.0    32
3.0       31
0.0       29
1.0       16
Name: count, dtype: int64

Collapsing smoking variable...
Smoking collapsed - Value counts:
smoking_collapsed
1.0       47
9999.0    32
0.0       29
Name: count, dtype: int64

Performing MICE imputation for smoking=9999...
Running MICE on 108 unique patients
Initialized logger with name mice 1-3
Dataset 0
1  | smoking_collapsed_for_imputation
2  | smoking_collapsed_for_imputation
3  | smoking_collapsed_for_imputation
Dataset 1
1  | smoking_collapsed_for_imputation
2  | smoking_collapsed_for_imputation
3  | smoking_collapsed_for_imputation
Dataset 2
1  | smoking_collapsed_for_imputation
2  | smoking_collapsed_for_imputation
3  | smoking_collapsed_for_imputation
Smoking after MICE imputation - Value coun

In [101]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,fwb_subscale_score_12mo,ecs_subscale_score_Baseline,ecs_subscale_score_12mo,fact_e_total_Baseline_std,bmi,bmi_std,smoking_collapsed_imputed
id,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,25.0,46.0,55.0,-0.485421,46.075514,3.300320,1
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,10.0,36.0,48.0,-1.448897,25.360019,-0.226055,1
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,13.0,52.0,54.0,-0.399778,34.213824,1.281119,1
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,22.0,37.0,57.0,-0.421189,27.115631,0.072801,0
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,20.5,57.0,51.0,0.178308,32.922429,1.061286,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,16.0,67.0,38.0,0.906268,27.297959,0.103839,1
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,10.0,58.0,35.0,1.034731,33.217993,1.111599,0
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,15.0,62.0,37.0,0.520877,39.100346,2.112946,1


In [102]:
smoking_summaries

{'n_total': 108,
 'n_missing': 26,
 'n_never': 40,
 'n_ever': 68,
 'mean': 0.6296296296296297,
 'sd': 0.4851551942197118,
 'preprocessed_data':            baseline_type  fact_e_total_Baseline  fact_e_total_12mo  \
 id                                                                   
 32        baseline_arm_1                  117.5              159.0   
 76    preoperative_arm_1                   95.0               97.0   
 125       baseline_arm_1                  119.5              129.0   
 160       baseline_arm_1                  119.0              152.0   
 198       baseline_arm_1                  133.0              139.0   
 ...                  ...                    ...                ...   
 1566  preoperative_arm_1                  150.0              107.0   
 1571      baseline_arm_1                  153.0              104.0   
 1572      baseline_arm_1                  141.0              115.0   
 1615      baseline_arm_1                  126.0              111.0   
 1653

# Characteristic by class - 3) alcohol consumption

In [103]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,fwb_subscale_score_12mo,ecs_subscale_score_Baseline,ecs_subscale_score_12mo,fact_e_total_Baseline_std,bmi,bmi_std,smoking_collapsed_imputed
id,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,25.0,46.0,55.0,-0.485421,46.075514,3.300320,1
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,10.0,36.0,48.0,-1.448897,25.360019,-0.226055,1
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,13.0,52.0,54.0,-0.399778,34.213824,1.281119,1
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,22.0,37.0,57.0,-0.421189,27.115631,0.072801,0
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,20.5,57.0,51.0,0.178308,32.922429,1.061286,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,16.0,67.0,38.0,0.906268,27.297959,0.103839,1
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,10.0,58.0,35.0,1.034731,33.217993,1.111599,0
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,15.0,62.0,37.0,0.520877,39.100346,2.112946,1


In [104]:
def preprocess_alcohol(score_comparison, filtered_df):
    """
    Preprocess alcohol: collapse categories + MICE imputation for missing
    Alcohol is categorical: 0=Non-drinker, 1=Ever drank (collapsed from 1,2), 9999=Unknown (then imputed with MICE)
    
    Returns:
    - Dictionary with preprocessing results and statistics
    """
    import miceforest as mf
    
    print("="*80)
    print("STEP 2: PREDICTOR PREPROCESSING - Alcohol")
    print("="*80)
    print("Preparing alcohol data for MICE imputation...")
    
    # Get all unique patient IDs from score_comparison
    all_patient_ids = score_comparison.index.tolist()
    
    # Merge with filtered_df to get alcohol
    alcohol_prep = pd.DataFrame({'patient_id': all_patient_ids})
    alcohol_prep = alcohol_prep.merge(
        filtered_df[["id", "alcohol"]],
        left_on='patient_id',
        right_on='id',
        how='left'
    )
    
    print(alcohol_prep['alcohol'].value_counts())
    
    # Find patient IDs where all alcohol values are missing
    patients_all_missing = alcohol_prep.groupby('patient_id')['alcohol'].apply(lambda x: x.isna().all())
    patients_to_impute = patients_all_missing[patients_all_missing].index
    
    # Impute alcohol for those patients with 9999 (unknown)
    mask = alcohol_prep['patient_id'].isin(patients_to_impute)
    alcohol_prep.loc[mask, 'alcohol'] = 9999.0
    print(f"Imputed {len(patients_to_impute)} unique patients with alcohol=9999 (unknown)")
    
    # Remove NA and duplicates
    alcohol_prep_clean = alcohol_prep.dropna(subset=["alcohol"])
    alcohol_prep_clean = alcohol_prep_clean.drop_duplicates()
    
    print(alcohol_prep_clean['alcohol'].value_counts())
    
    # Collapse alcohol variable
    print("\nCollapsing alcohol variable...")
    alcohol_prep_clean['alcohol_collapsed'] = alcohol_prep_clean['alcohol'].copy()
    alcohol_prep_clean.loc[alcohol_prep_clean['alcohol'].isin([1, 2]), 'alcohol_collapsed'] = 1  # Ever Drank
    alcohol_prep_clean.loc[alcohol_prep_clean['alcohol'] == 0, 'alcohol_collapsed'] = 0  # Non-Drinker
    alcohol_prep_clean.loc[alcohol_prep_clean['alcohol'] == 9999, 'alcohol_collapsed'] = 9999  # Unknown
    
    print("Alcohol collapsed - Value counts:")
    print(alcohol_prep_clean['alcohol_collapsed'].value_counts())
    
    # Perform MICE imputation ONCE
    print("\nPerforming MICE imputation for alcohol=9999...")
    
    # Replace 9999 with NaN for imputation
    alcohol_prep_clean['alcohol_collapsed_for_imputation'] = alcohol_prep_clean['alcohol_collapsed'].replace(9999, np.nan)
    
    # Merge baseline FACT-E for imputation model
    alcohol_prep_clean = alcohol_prep_clean.merge(
        score_comparison[['fact_e_total_Baseline']],
        left_on='patient_id',
        right_index=True,
        how='left'
    )
    
    print(f"Running MICE on {len(alcohol_prep_clean)} unique patients")
    
    # Create dataset for MICE
    mice_data = alcohol_prep_clean[['alcohol_collapsed_for_imputation', 'fact_e_total_Baseline']].copy()
    
    # MICE imputation
    mice_imputer = mf.ImputationKernel(
        mice_data,
        datasets=3,
        random_state=42
    )
    
    mice_imputer.mice(iterations=3, verbose=True)
    
    # Get completed dataset
    completed_data = mice_imputer.complete_data(0)
    
    # Extract imputed values
    imputed_alcohol = completed_data['alcohol_collapsed_for_imputation'].round().astype(int)
    
    # Create mapping dictionary for all patients
    alcohol_imputation_map = dict(zip(alcohol_prep_clean['patient_id'], imputed_alcohol))
    
    print("Alcohol after MICE imputation - Value counts:")
    print(pd.Series(imputed_alcohol).value_counts())
    print("\n" + "="*80)
    
    # Create final dataframe with imputed alcohol values
    alcohol_final = pd.DataFrame({
        'patient_id': list(alcohol_imputation_map.keys()),
        'alcohol_collapsed_imputed': list(alcohol_imputation_map.values())
    })
    alcohol_final = alcohol_final.set_index('patient_id')
    
    # Calculate summary statistics
    alcohol_data = alcohol_final['alcohol_collapsed_imputed']
    n_non_drinker = (alcohol_data == 0).sum()
    n_ever_drank = (alcohol_data == 1).sum()
    total = len(alcohol_data)
    
    print(f"\nFinal alcohol distribution:")
    print(f"  N: {total}")
    print(f"  Non-drinker (0): {n_non_drinker} ({n_non_drinker/total*100:.1f}%)")
    print(f"  Ever drank (1): {n_ever_drank} ({n_ever_drank/total*100:.1f}%)")
    print(f"  Mean: {alcohol_data.mean():.2f}")
    print(f"  SD: {alcohol_data.std():.2f}")
    
    # Merge back to score_comparison
    score_comparison = score_comparison.merge(
        alcohol_final[['alcohol_collapsed_imputed']], 
        left_index=True, 
        right_index=True, 
        how='left'
    )
    
    print("\n✓ Created column: 'alcohol_collapsed_imputed'")
    print("="*80)
    
    # Return results dictionary
    results = {
        'n_total': len(all_patient_ids),
        'n_missing': len(patients_to_impute),
        'n_non_drinker': n_non_drinker,
        'n_ever_drank': n_ever_drank,
        'mean': alcohol_data.mean(),
        'sd': alcohol_data.std(),
        'preprocessed_data': score_comparison
    }
    
    return results

# Run preprocessing
alcohol_summaries = preprocess_alcohol(score_comparison, filtered_df)

# Update score_comparison with preprocessed data
score_comparison = alcohol_summaries['preprocessed_data']

# Display first few rows to verify
print("\nFirst few rows of alcohol (imputed):")
print(score_comparison[['alcohol_collapsed_imputed']].head(10))

STEP 2: PREDICTOR PREPROCESSING - Alcohol
Preparing alcohol data for MICE imputation...
alcohol
1.0       40
0.0       27
9999.0     9
2.0        6
Name: count, dtype: int64
Imputed 26 unique patients with alcohol=9999 (unknown)
alcohol
1.0       40
9999.0    35
0.0       27
2.0        6
Name: count, dtype: int64

Collapsing alcohol variable...
Alcohol collapsed - Value counts:
alcohol_collapsed
1.0       46
9999.0    35
0.0       27
Name: count, dtype: int64

Performing MICE imputation for alcohol=9999...
Running MICE on 108 unique patients
Initialized logger with name mice 1-3
Dataset 0
1  | alcohol_collapsed_for_imputation
2  | alcohol_collapsed_for_imputation
3  | alcohol_collapsed_for_imputation
Dataset 1
1  | alcohol_collapsed_for_imputation
2  | alcohol_collapsed_for_imputation
3  | alcohol_collapsed_for_imputation
Dataset 2
1  | alcohol_collapsed_for_imputation
2  | alcohol_collapsed_for_imputation
3  | alcohol_collapsed_for_imputation
Alcohol after MICE imputation - Value coun

In [105]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,fwb_subscale_score_12mo,ecs_subscale_score_Baseline,ecs_subscale_score_12mo,fact_e_total_Baseline_std,bmi,bmi_std,smoking_collapsed_imputed,alcohol_collapsed_imputed
id,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,25.0,46.0,55.0,-0.485421,46.075514,3.300320,1,1
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,10.0,36.0,48.0,-1.448897,25.360019,-0.226055,1,1
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,13.0,52.0,54.0,-0.399778,34.213824,1.281119,1,0
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,22.0,37.0,57.0,-0.421189,27.115631,0.072801,0,0
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,20.5,57.0,51.0,0.178308,32.922429,1.061286,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,16.0,67.0,38.0,0.906268,27.297959,0.103839,1,1
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,10.0,58.0,35.0,1.034731,33.217993,1.111599,0,0
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,15.0,62.0,37.0,0.520877,39.100346,2.112946,1,1


# Characteristic by class - 4) level of tumor (level_tumor)

In [107]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,fwb_subscale_score_12mo,ecs_subscale_score_Baseline,ecs_subscale_score_12mo,fact_e_total_Baseline_std,bmi,bmi_std,smoking_collapsed_imputed,alcohol_collapsed_imputed
id,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,25.0,46.0,55.0,-0.485421,46.075514,3.300320,1,1
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,10.0,36.0,48.0,-1.448897,25.360019,-0.226055,1,1
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,13.0,52.0,54.0,-0.399778,34.213824,1.281119,1,0
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,22.0,37.0,57.0,-0.421189,27.115631,0.072801,0,0
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,20.5,57.0,51.0,0.178308,32.922429,1.061286,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,16.0,67.0,38.0,0.906268,27.297959,0.103839,1,1
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,10.0,58.0,35.0,1.034731,33.217993,1.111599,0,0
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,15.0,62.0,37.0,0.520877,39.100346,2.112946,1,1


In [108]:
def preprocess_level_tumor(score_comparison, filtered_df):
    """
    Preprocess level_tumor: collapse categories + MICE imputation for missing
    Tumor level is categorical: 1=Cardia/GEJ (5,6,7), 2=Lower Esophagus (4), 3=Middle/Upper Esophagus (1,2,3), 9999=Unknown (then imputed with MICE)
    
    Returns:
    - Dictionary with preprocessing results and statistics
    """
    import miceforest as mf
    
    print("="*80)
    print("STEP 2: PREDICTOR PREPROCESSING - Tumor Level")
    print("="*80)
    print("Preparing tumor level data for MICE imputation...")
    
    # Get all unique patient IDs from score_comparison
    all_patient_ids = score_comparison.index.tolist()
    
    # Merge with filtered_df to get level_tumor
    level_prep = pd.DataFrame({'patient_id': all_patient_ids})
    level_prep = level_prep.merge(
        filtered_df[["id", "level_tumor"]],
        left_on='patient_id',
        right_on='id',
        how='left'
    )
    
    print(level_prep['level_tumor'].value_counts())
    
    # Find patient IDs where all level_tumor values are missing
    patients_all_missing = level_prep.groupby('patient_id')['level_tumor'].apply(lambda x: x.isna().all())
    patients_to_impute = patients_all_missing[patients_all_missing].index
    
    # Impute level_tumor for those patients with 9999 (unknown)
    mask = level_prep['patient_id'].isin(patients_to_impute)
    level_prep.loc[mask, 'level_tumor'] = 9999.0
    print(f"Imputed {len(patients_to_impute)} unique patients with level_tumor=9999 (unknown)")
    
    # Remove NA and duplicates
    level_prep_clean = level_prep.dropna(subset=["level_tumor"])
    level_prep_clean = level_prep_clean.drop_duplicates()
    
    print(level_prep_clean['level_tumor'].value_counts())
    
    # Collapse level_tumor variable
    print("\nCollapsing level_tumor variable...")
    level_prep_clean['level_tumor_collapsed'] = level_prep_clean['level_tumor'].copy()
    level_prep_clean.loc[level_prep_clean['level_tumor'].isin([5, 6, 7]), 'level_tumor_collapsed'] = 1  # Cardia/GEJ
    level_prep_clean.loc[level_prep_clean['level_tumor'] == 4, 'level_tumor_collapsed'] = 2  # Lower Esophagus
    level_prep_clean.loc[level_prep_clean['level_tumor'].isin([1, 2, 3]), 'level_tumor_collapsed'] = 3  # Middle or Upper Esophagus
    level_prep_clean.loc[level_prep_clean['level_tumor'] == 9999, 'level_tumor_collapsed'] = 9999  # Unknown
    
    print("Level tumor collapsed - Value counts:")
    print(level_prep_clean['level_tumor_collapsed'].value_counts())
    
    # Perform MICE imputation ONCE
    print("\nPerforming MICE imputation for level_tumor=9999...")
    
    # Replace 9999 with NaN for imputation
    level_prep_clean['level_tumor_collapsed_for_imputation'] = level_prep_clean['level_tumor_collapsed'].replace(9999, np.nan)
    
    # Merge baseline FACT-E for imputation model
    level_prep_clean = level_prep_clean.merge(
        score_comparison[['fact_e_total_Baseline']],
        left_on='patient_id',
        right_index=True,
        how='left'
    )
    
    print(f"Running MICE on {len(level_prep_clean)} unique patients")
    
    # Create dataset for MICE
    mice_data = level_prep_clean[['level_tumor_collapsed_for_imputation', 'fact_e_total_Baseline']].copy()
    
    # MICE imputation
    mice_imputer = mf.ImputationKernel(
        mice_data,
        datasets=3,
        random_state=42
    )
    
    mice_imputer.mice(iterations=3, verbose=True)
    
    # Get completed dataset
    completed_data = mice_imputer.complete_data(0)
    
    # Extract imputed values
    imputed_level = completed_data['level_tumor_collapsed_for_imputation'].round().astype(int)
    
    # Create mapping dictionary for all patients
    level_imputation_map = dict(zip(level_prep_clean['patient_id'], imputed_level))
    
    print("Level tumor after MICE imputation - Value counts:")
    print(pd.Series(imputed_level).value_counts())
    print("\n" + "="*80)
    
    # Create final dataframe with imputed level_tumor values
    level_final = pd.DataFrame({
        'patient_id': list(level_imputation_map.keys()),
        'level_tumor_collapsed_imputed': list(level_imputation_map.values())
    })
    level_final = level_final.set_index('patient_id')
    
    # Calculate summary statistics
    level_data = level_final['level_tumor_collapsed_imputed']
    n_cardia_gej = (level_data == 1).sum()
    n_lower_esoph = (level_data == 2).sum()
    n_middle_upper = (level_data == 3).sum()
    total = len(level_data)
    
    print(f"\nFinal tumor level distribution:")
    print(f"  N: {total}")
    print(f"  Cardia/GEJ (1): {n_cardia_gej} ({n_cardia_gej/total*100:.1f}%)")
    print(f"  Lower Esoph (2): {n_lower_esoph} ({n_lower_esoph/total*100:.1f}%)")
    print(f"  Middle/Upper (3): {n_middle_upper} ({n_middle_upper/total*100:.1f}%)")
    print(f"  Mean: {level_data.mean():.2f}")
    print(f"  SD: {level_data.std():.2f}")
    
    # Merge back to score_comparison
    score_comparison = score_comparison.merge(
        level_final[['level_tumor_collapsed_imputed']], 
        left_index=True, 
        right_index=True, 
        how='left'
    )
    
    print("\n✓ Created column: 'level_tumor_collapsed_imputed'")
    print("="*80)
    
    # Return results dictionary
    results = {
        'n_total': len(all_patient_ids),
        'n_missing': len(patients_to_impute),
        'n_cardia_gej': n_cardia_gej,
        'n_lower_esoph': n_lower_esoph,
        'n_middle_upper': n_middle_upper,
        'mean': level_data.mean(),
        'sd': level_data.std(),
        'preprocessed_data': score_comparison
    }
    
    return results

# Run preprocessing
level_tumor_summaries = preprocess_level_tumor(score_comparison, filtered_df)

# Update score_comparison with preprocessed data
score_comparison = level_tumor_summaries['preprocessed_data']

# Display first few rows to verify
print("\nFirst few rows of level_tumor (imputed):")
print(score_comparison[['level_tumor_collapsed_imputed']].head(10))

STEP 2: PREDICTOR PREPROCESSING - Tumor Level
Preparing tumor level data for MICE imputation...
level_tumor
4.0    47
6.0    28
7.0    15
5.0     9
3.0     8
2.0     2
1.0     1
Name: count, dtype: int64
Imputed 1 unique patients with level_tumor=9999 (unknown)
level_tumor
4.0       47
6.0       26
7.0       15
5.0        9
3.0        7
2.0        2
9999.0     1
1.0        1
Name: count, dtype: int64

Collapsing level_tumor variable...
Level tumor collapsed - Value counts:
level_tumor_collapsed
1.0       50
2.0       47
3.0       10
9999.0     1
Name: count, dtype: int64

Performing MICE imputation for level_tumor=9999...
Running MICE on 108 unique patients
Initialized logger with name mice 1-3
Dataset 0
1  | level_tumor_collapsed_for_imputation
2  | level_tumor_collapsed_for_imputation
3  | level_tumor_collapsed_for_imputation
Dataset 1
1  | level_tumor_collapsed_for_imputation
2  | level_tumor_collapsed_for_imputation
3  | level_tumor_collapsed_for_imputation
Dataset 2
1  | level_tum

In [109]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,fwb_subscale_score_12mo,ecs_subscale_score_Baseline,ecs_subscale_score_12mo,fact_e_total_Baseline_std,bmi,bmi_std,smoking_collapsed_imputed,alcohol_collapsed_imputed,level_tumor_collapsed_imputed
id,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,25.0,46.0,55.0,-0.485421,46.075514,3.300320,1,1,2
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,10.0,36.0,48.0,-1.448897,25.360019,-0.226055,1,1,2
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,13.0,52.0,54.0,-0.399778,34.213824,1.281119,1,0,2
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,22.0,37.0,57.0,-0.421189,27.115631,0.072801,0,0,1
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,20.5,57.0,51.0,0.178308,32.922429,1.061286,1,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,16.0,67.0,38.0,0.906268,27.297959,0.103839,1,1,1
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,10.0,58.0,35.0,1.034731,33.217993,1.111599,0,0,1
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,15.0,62.0,37.0,0.520877,39.100346,2.112946,1,1,1


# Characteristic by class - 5) histology (path_histology)

In [113]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,fwb_subscale_score_12mo,ecs_subscale_score_Baseline,ecs_subscale_score_12mo,fact_e_total_Baseline_std,bmi,bmi_std,smoking_collapsed_imputed,alcohol_collapsed_imputed,level_tumor_collapsed_imputed
id,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,25.0,46.0,55.0,-0.485421,46.075514,3.300320,1,1,2
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,10.0,36.0,48.0,-1.448897,25.360019,-0.226055,1,1,2
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,13.0,52.0,54.0,-0.399778,34.213824,1.281119,1,0,2
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,22.0,37.0,57.0,-0.421189,27.115631,0.072801,0,0,1
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,20.5,57.0,51.0,0.178308,32.922429,1.061286,1,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,16.0,67.0,38.0,0.906268,27.297959,0.103839,1,1,1
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,10.0,58.0,35.0,1.034731,33.217993,1.111599,0,0,1
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,15.0,62.0,37.0,0.520877,39.100346,2.112946,1,1,1


In [114]:
def preprocess_histology(score_comparison, filtered_df):
    """
    Preprocess histology: collapse categories + MICE imputation for missing
    Histology is categorical: 0=No carcinoma (1), 1=Adenocarcinoma (4), 2=SCC (5), 3=Other (2,3,6,7,8), 9999=Unknown (then imputed with MICE)
    
    Returns:
    - Dictionary with preprocessing results and statistics
    """
    import miceforest as mf
    
    print("="*80)
    print("STEP 2: PREDICTOR PREPROCESSING - Histology")
    print("="*80)
    print("Preparing histology data for MICE imputation...")
    
    # Get all unique patient IDs from score_comparison
    all_patient_ids = score_comparison.index.tolist()
    
    # Merge with filtered_df to get histology columns
    histology_prep = pd.DataFrame({'patient_id': all_patient_ids})
    histology_prep = histology_prep.merge(
        filtered_df[["id", "path_histology___noca", "path_histology___hgd", "path_histology___cis", 
                        "path_histology___adeno", "path_histology___scc", "path_histology___nec", 
                        "path_histology___other"]],
        left_on='patient_id',
        right_on='id',
        how='left'
    )
    
    # Create histology mapping
    histology_mapping = {
        "path_histology___noca": 1,
        "path_histology___hgd": 2,
        "path_histology___cis": 3,
        "path_histology___adeno": 4,
        "path_histology___scc": 5,
        "path_histology___nec": 6,
        "path_histology___other": 7
    }
    
    # Assign histology
    def assign_histology(row):
        active_cols = [col for col in histology_mapping.keys() if row[col] == 1]
        if len(active_cols) == 1:
            return histology_mapping[active_cols[0]]
        elif len(active_cols) > 1:
            return 8  # mixed
        else:
            return None  # missing/NaN
    
    histology_prep['histology'] = histology_prep.apply(assign_histology, axis=1)
    
    # Group by patient_id and check if patient has multiple different histology types
    patient_histology = histology_prep.groupby('patient_id')['histology'].apply(
        lambda x: 8 if len(set(x.dropna())) > 1 else x.dropna().iloc[0] if len(x.dropna()) > 0 else None
    ).reset_index()
    patient_histology.columns = ['patient_id', 'final_histology']
    
    # Merge back
    histology_prep = histology_prep.merge(patient_histology[['patient_id', 'final_histology']], on='patient_id', how='left')
    
    # Drop the individual histology columns and rename final_histology
    histology_prep = histology_prep.drop(columns=[
        "path_histology___noca",
        "path_histology___hgd",
        "path_histology___cis",
        "path_histology___adeno",
        "path_histology___scc",
        "path_histology___nec",
        "path_histology___other",
        "histology"
    ]).rename(columns={"final_histology": "path_histology"})
    
    print(histology_prep['path_histology'].value_counts())
    
    # Find patient IDs where all path_histology values are missing
    patients_all_missing = histology_prep.groupby('patient_id')['path_histology'].apply(lambda x: x.isna().all())
    patients_to_impute = patients_all_missing[patients_all_missing].index
    
    # Impute path_histology for those patients with 9999 (unknown)
    mask = histology_prep['patient_id'].isin(patients_to_impute)
    histology_prep.loc[mask, 'path_histology'] = 9999.0
    print(f"Imputed {len(patients_to_impute)} unique patients with path_histology=9999 (unknown)")
    
    # Create a clean dataframe with no missing values in path_histology
    histology_prep_clean = histology_prep.dropna(subset=["path_histology"])
    histology_prep_clean = histology_prep_clean.drop_duplicates()
    
    print(histology_prep_clean['path_histology'].value_counts())
    
    # Collapse path_histology variable
    print("\nCollapsing path_histology variable...")
    histology_prep_clean['path_histology_collapsed'] = histology_prep_clean['path_histology'].copy()
    histology_prep_clean.loc[histology_prep_clean['path_histology'] == 1, 'path_histology_collapsed'] = 0  # No carcinoma
    histology_prep_clean.loc[histology_prep_clean['path_histology'] == 4, 'path_histology_collapsed'] = 1  # Adenocarcinoma
    histology_prep_clean.loc[histology_prep_clean['path_histology'] == 5, 'path_histology_collapsed'] = 2  # SCC
    # All other options except 9999 become "Other"
    other_mask = (~histology_prep_clean['path_histology'].isin([1, 4, 5, 9999])) & (~histology_prep_clean['path_histology'].isna())
    histology_prep_clean.loc[other_mask, 'path_histology_collapsed'] = 3  # Other
    histology_prep_clean.loc[histology_prep_clean['path_histology'] == 9999, 'path_histology_collapsed'] = 9999  # Unknown
    
    print("Path histology collapsed - Value counts:")
    print(histology_prep_clean['path_histology_collapsed'].value_counts())
    
    # Perform MICE imputation ONCE
    print("\nPerforming MICE imputation for path_histology=9999...")
    
    # Replace 9999 with NaN for imputation
    histology_prep_clean['path_histology_collapsed_for_imputation'] = histology_prep_clean['path_histology_collapsed'].replace(9999, np.nan)
    
    # Merge baseline FACT-E for imputation model
    histology_prep_clean = histology_prep_clean.merge(
        score_comparison[['fact_e_total_Baseline']],
        left_on='patient_id',
        right_index=True,
        how='left'
    )
    
    print(f"Running MICE on {len(histology_prep_clean)} unique patients")
    
    # Create dataset for MICE
    mice_data = histology_prep_clean[['path_histology_collapsed_for_imputation', 'fact_e_total_Baseline']].copy()
    
    # MICE imputation
    mice_imputer = mf.ImputationKernel(
        mice_data,
        datasets=3,
        random_state=42
    )
    
    mice_imputer.mice(iterations=3, verbose=True)
    
    # Get completed dataset
    completed_data = mice_imputer.complete_data(0)
    
    # Extract imputed values
    imputed_histology = completed_data['path_histology_collapsed_for_imputation'].round().astype(int)
    
    # Create mapping dictionary for all patients
    histology_imputation_map = dict(zip(histology_prep_clean['patient_id'], imputed_histology))
    
    print("Path histology after MICE imputation - Value counts:")
    print(pd.Series(imputed_histology).value_counts())
    print("\n" + "="*80)
    
    # Create final dataframe with imputed histology values
    histology_final = pd.DataFrame({
        'patient_id': list(histology_imputation_map.keys()),
        'path_histology_collapsed_imputed': list(histology_imputation_map.values())
    })
    histology_final = histology_final.set_index('patient_id')
    
    # Calculate summary statistics
    histology_data = histology_final['path_histology_collapsed_imputed']
    n_no_carcinoma = (histology_data == 0).sum()
    n_adeno = (histology_data == 1).sum()
    n_scc = (histology_data == 2).sum()
    n_other = (histology_data == 3).sum()
    total = len(histology_data)
    
    print(f"\nFinal histology distribution:")
    print(f"  N: {total}")
    print(f"  No Carcinoma (0): {n_no_carcinoma} ({n_no_carcinoma/total*100:.1f}%)")
    print(f"  Adenocarcinoma (1): {n_adeno} ({n_adeno/total*100:.1f}%)")
    print(f"  SCC (2): {n_scc} ({n_scc/total*100:.1f}%)")
    print(f"  Other (3): {n_other} ({n_other/total*100:.1f}%)")
    print(f"  Mean: {histology_data.mean():.2f}")
    print(f"  SD: {histology_data.std():.2f}")
    
    # Merge back to score_comparison
    score_comparison = score_comparison.merge(
        histology_final[['path_histology_collapsed_imputed']], 
        left_index=True, 
        right_index=True, 
        how='left'
    )
    
    print("\n✓ Created column: 'path_histology_collapsed_imputed'")
    print("="*80)
    
    # Return results dictionary
    results = {
        'n_total': len(all_patient_ids),
        'n_missing': len(patients_to_impute),
        'n_no_carcinoma': n_no_carcinoma,
        'n_adeno': n_adeno,
        'n_scc': n_scc,
        'n_other': n_other,
        'mean': histology_data.mean(),
        'sd': histology_data.std(),
        'preprocessed_data': score_comparison
    }
    
    return results

# Run preprocessing
histology_summaries = preprocess_histology(score_comparison, filtered_df)

# Update score_comparison with preprocessed data
score_comparison = histology_summaries['preprocessed_data']

# Display first few rows to verify
print("\nFirst few rows of histology (imputed):")
print(score_comparison[['path_histology_collapsed_imputed']].head(10))

STEP 2: PREDICTOR PREPROCESSING - Histology
Preparing histology data for MICE imputation...
path_histology
4.0    906
8.0    280
1.0    158
5.0    143
7.0     10
2.0     10
Name: count, dtype: int64
Imputed 1 unique patients with path_histology=9999 (unknown)
path_histology
4.0       63
8.0       20
1.0       12
5.0       10
7.0        1
9999.0     1
2.0        1
Name: count, dtype: int64

Collapsing path_histology variable...
Path histology collapsed - Value counts:
path_histology_collapsed
1.0       63
3.0       22
0.0       12
2.0       10
9999.0     1
Name: count, dtype: int64

Performing MICE imputation for path_histology=9999...
Running MICE on 108 unique patients
Initialized logger with name mice 1-3
Dataset 0
1  | path_histology_collapsed_for_imputation
2  | path_histology_collapsed_for_imputation
3  | path_histology_collapsed_for_imputation
Dataset 1
1  | path_histology_collapsed_for_imputation
2  | path_histology_collapsed_for_imputation
3  | path_histology_collapsed_for_impu

In [115]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,fwb_subscale_score_12mo,ecs_subscale_score_Baseline,ecs_subscale_score_12mo,fact_e_total_Baseline_std,bmi,bmi_std,smoking_collapsed_imputed,alcohol_collapsed_imputed,level_tumor_collapsed_imputed,path_histology_collapsed_imputed
id,,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,25.0,46.0,55.0,-0.485421,46.075514,3.300320,1,1,2,0
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,10.0,36.0,48.0,-1.448897,25.360019,-0.226055,1,1,2,1
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,13.0,52.0,54.0,-0.399778,34.213824,1.281119,1,0,2,1
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,22.0,37.0,57.0,-0.421189,27.115631,0.072801,0,0,1,1
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,20.5,57.0,51.0,0.178308,32.922429,1.061286,1,1,2,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,16.0,67.0,38.0,0.906268,27.297959,0.103839,1,1,1,1
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,10.0,58.0,35.0,1.034731,33.217993,1.111599,0,0,1,0
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,15.0,62.0,37.0,0.520877,39.100346,2.112946,1,1,1,3


# Characteristic by class - 6) pathological stage (path_esoph_stage)

In [118]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,fwb_subscale_score_12mo,ecs_subscale_score_Baseline,ecs_subscale_score_12mo,fact_e_total_Baseline_std,bmi,bmi_std,smoking_collapsed_imputed,alcohol_collapsed_imputed,level_tumor_collapsed_imputed,path_histology_collapsed_imputed
id,,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,25.0,46.0,55.0,-0.485421,46.075514,3.300320,1,1,2,0
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,10.0,36.0,48.0,-1.448897,25.360019,-0.226055,1,1,2,1
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,13.0,52.0,54.0,-0.399778,34.213824,1.281119,1,0,2,1
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,22.0,37.0,57.0,-0.421189,27.115631,0.072801,0,0,1,1
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,20.5,57.0,51.0,0.178308,32.922429,1.061286,1,1,2,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,16.0,67.0,38.0,0.906268,27.297959,0.103839,1,1,1,1
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,10.0,58.0,35.0,1.034731,33.217993,1.111599,0,0,1,0
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,15.0,62.0,37.0,0.520877,39.100346,2.112946,1,1,1,3


In [119]:
def preprocess_path_stage(score_comparison, filtered_df):
    """
    Preprocess path_esoph_stage: collapse categories + MICE imputation for missing
    Path stage is categorical: 0=Stage 0, 1=Stage I (1a,1b,1c), 2=Stage II (2a,2b), 3=Stage III (3a,3b), 4=Stage IV (4a,4b), 5=pCR, 9999=Unknown (then imputed with MICE)
    
    Returns:
    - Dictionary with preprocessing results and statistics
    """
    import miceforest as mf
    
    print("="*80)
    print("STEP 2: PREDICTOR PREPROCESSING - Pathological Stage")
    print("="*80)
    print("Preparing pathological stage data for MICE imputation...")
    
    # Get all unique patient IDs from score_comparison
    all_patient_ids = score_comparison.index.tolist()
    
    # Merge with filtered_df to get path_esoph_stage
    stage_prep = pd.DataFrame({'patient_id': all_patient_ids})
    stage_prep = stage_prep.merge(
        filtered_df[["id", "path_esoph_stage"]],
        left_on='patient_id',
        right_on='id',
        how='left'
    )
    
    print(stage_prep['path_esoph_stage'].value_counts())
    
    # Find patient IDs where all path_esoph_stage values are missing
    patients_all_missing = stage_prep.groupby('patient_id')['path_esoph_stage'].apply(lambda x: x.isna().all())
    patients_to_impute = patients_all_missing[patients_all_missing].index
    
    # Impute path_esoph_stage for those patients with 9999 (unknown)
    mask = stage_prep['patient_id'].isin(patients_to_impute)
    stage_prep.loc[mask, 'path_esoph_stage'] = 9999.0
    print(f"Imputed {len(patients_to_impute)} unique patients with path_esoph_stage=9999 (unknown)")
    
    # Create a clean dataframe with no missing values in path_esoph_stage
    stage_prep_clean = stage_prep.dropna(subset=["path_esoph_stage"])
    stage_prep_clean = stage_prep_clean.drop_duplicates()
    
    print(stage_prep_clean['path_esoph_stage'].value_counts())
    
    # Collapse path_esoph_stage variable
    print("\nCollapsing path_esoph_stage variable...")
    stage_prep_clean['path_esoph_stage_collapsed'] = stage_prep_clean['path_esoph_stage'].copy()
    stage_prep_clean.loc[stage_prep_clean['path_esoph_stage'] == 0, 'path_esoph_stage_collapsed'] = 0  # Stage 0
    stage_prep_clean.loc[stage_prep_clean['path_esoph_stage'].isin(['1a', '1b', '1c']), 'path_esoph_stage_collapsed'] = 1  # Stage I
    stage_prep_clean.loc[stage_prep_clean['path_esoph_stage'].isin(['2a', '2b']), 'path_esoph_stage_collapsed'] = 2  # Stage II
    stage_prep_clean.loc[stage_prep_clean['path_esoph_stage'].isin(['3a', '3b']), 'path_esoph_stage_collapsed'] = 3  # Stage III
    stage_prep_clean.loc[stage_prep_clean['path_esoph_stage'].isin(['4a', '4b']), 'path_esoph_stage_collapsed'] = 4  # Stage IV
    stage_prep_clean.loc[stage_prep_clean['path_esoph_stage'] == 5, 'path_esoph_stage_collapsed'] = 5  # pCR
    stage_prep_clean.loc[stage_prep_clean['path_esoph_stage'] == 9999, 'path_esoph_stage_collapsed'] = 9999  # Unknown
    
    stage_prep_clean['path_esoph_stage_collapsed'] = pd.to_numeric(stage_prep_clean['path_esoph_stage_collapsed'], errors='coerce')
    
    print("Path esoph stage collapsed - Value counts:")
    print(stage_prep_clean['path_esoph_stage_collapsed'].value_counts())
    
    # Perform MICE imputation ONCE
    print("\nPerforming MICE imputation for path_esoph_stage=9999...")
    
    # Replace 9999 with NaN for imputation
    stage_prep_clean['path_esoph_stage_collapsed_for_imputation'] = stage_prep_clean['path_esoph_stage_collapsed'].replace(9999, np.nan)
    
    # Merge baseline FACT-E for imputation model
    stage_prep_clean = stage_prep_clean.merge(
        score_comparison[['fact_e_total_Baseline']],
        left_on='patient_id',
        right_index=True,
        how='left'
    )
    
    print(f"Running MICE on {len(stage_prep_clean)} unique patients")
    
    # Create dataset for MICE
    mice_data = stage_prep_clean[['path_esoph_stage_collapsed_for_imputation', 'fact_e_total_Baseline']].copy()
    
    # MICE imputation
    mice_imputer = mf.ImputationKernel(
        mice_data,
        datasets=3,
        random_state=42
    )
    
    mice_imputer.mice(iterations=3, verbose=True)
    
    # Get completed dataset
    completed_data = mice_imputer.complete_data(0)
    
    # Extract imputed values
    imputed_stage = completed_data['path_esoph_stage_collapsed_for_imputation'].round().astype(int)
    
    # Create mapping dictionary for all patients
    stage_imputation_map = dict(zip(stage_prep_clean['patient_id'], imputed_stage))
    
    print("Path esoph stage after MICE imputation - Value counts:")
    print(pd.Series(imputed_stage).value_counts())
    print("\n" + "="*80)
    
    # Create final dataframe with imputed stage values
    stage_final = pd.DataFrame({
        'patient_id': list(stage_imputation_map.keys()),
        'path_esoph_stage_collapsed_imputed': list(stage_imputation_map.values())
    })
    stage_final = stage_final.set_index('patient_id')
    
    # Calculate summary statistics
    stage_data = stage_final['path_esoph_stage_collapsed_imputed']
    n_stage0 = (stage_data == 0).sum()
    n_stage1 = (stage_data == 1).sum()
    n_stage2 = (stage_data == 2).sum()
    n_stage3 = (stage_data == 3).sum()
    n_stage4 = (stage_data == 4).sum()
    n_pcr = (stage_data == 5).sum()
    total = len(stage_data)
    
    print(f"\nFinal pathological stage distribution:")
    print(f"  N: {total}")
    print(f"  Stage 0 (0): {n_stage0} ({n_stage0/total*100:.1f}%)")
    print(f"  Stage I (1): {n_stage1} ({n_stage1/total*100:.1f}%)")
    print(f"  Stage II (2): {n_stage2} ({n_stage2/total*100:.1f}%)")
    print(f"  Stage III (3): {n_stage3} ({n_stage3/total*100:.1f}%)")
    print(f"  Stage IV (4): {n_stage4} ({n_stage4/total*100:.1f}%)")
    print(f"  pCR (5): {n_pcr} ({n_pcr/total*100:.1f}%)")
    print(f"  Mean: {stage_data.mean():.2f}")
    print(f"  SD: {stage_data.std():.2f}")
    
    # Merge back to score_comparison
    score_comparison = score_comparison.merge(
        stage_final[['path_esoph_stage_collapsed_imputed']], 
        left_index=True, 
        right_index=True, 
        how='left'
    )
    
    print("\n✓ Created column: 'path_esoph_stage_collapsed_imputed'")
    print("="*80)
    
    # Return results dictionary
    results = {
        'n_total': len(all_patient_ids),
        'n_missing': len(patients_to_impute),
        'n_stage0': n_stage0,
        'n_stage1': n_stage1,
        'n_stage2': n_stage2,
        'n_stage3': n_stage3,
        'n_stage4': n_stage4,
        'n_pcr': n_pcr,
        'mean': stage_data.mean(),
        'sd': stage_data.std(),
        'preprocessed_data': score_comparison
    }
    
    return results

# Run preprocessing
path_stage_summaries = preprocess_path_stage(score_comparison, filtered_df)

# Update score_comparison with preprocessed data
score_comparison = path_stage_summaries['preprocessed_data']

# Display first few rows to verify
print("\nFirst few rows of path_esoph_stage (imputed):")
print(score_comparison[['path_esoph_stage_collapsed_imputed']].head(10))

STEP 2: PREDICTOR PREPROCESSING - Pathological Stage
Preparing pathological stage data for MICE imputation...
path_esoph_stage
3b    15
2b    12
0      6
3a     5
4a     4
2a     4
1b     3
5      2
4b     1
Name: count, dtype: int64
Imputed 58 unique patients with path_esoph_stage=9999 (unknown)
path_esoph_stage
9999.0    58
3b        13
2b        12
0          6
3a         5
4a         4
2a         4
1b         3
5          2
4b         1
Name: count, dtype: int64

Collapsing path_esoph_stage variable...
Path esoph stage collapsed - Value counts:
path_esoph_stage_collapsed
9999    58
3       18
2       16
0        6
4        5
1        3
5        2
Name: count, dtype: int64

Performing MICE imputation for path_esoph_stage=9999...
Running MICE on 108 unique patients
Initialized logger with name mice 1-3
Dataset 0
1  | path_esoph_stage_collapsed_for_imputation
2  | path_esoph_stage_collapsed_for_imputation
3  | path_esoph_stage_collapsed_for_imputation
Dataset 1
1  | path_esoph_stage_c

In [120]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,...,ecs_subscale_score_Baseline,ecs_subscale_score_12mo,fact_e_total_Baseline_std,bmi,bmi_std,smoking_collapsed_imputed,alcohol_collapsed_imputed,level_tumor_collapsed_imputed,path_histology_collapsed_imputed,path_esoph_stage_collapsed_imputed
id,,,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,...,46.0,55.0,-0.485421,46.075514,3.300320,1,1,2,0,4
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,...,36.0,48.0,-1.448897,25.360019,-0.226055,1,1,2,1,2
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,...,52.0,54.0,-0.399778,34.213824,1.281119,1,0,2,1,3
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,...,37.0,57.0,-0.421189,27.115631,0.072801,0,0,1,1,3
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,...,57.0,51.0,0.178308,32.922429,1.061286,1,1,2,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,...,67.0,38.0,0.906268,27.297959,0.103839,1,1,1,1,3
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,...,58.0,35.0,1.034731,33.217993,1.111599,0,0,1,0,2
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,...,62.0,37.0,0.520877,39.100346,2.112946,1,1,1,3,3


# Characteristic by class - 7) surgery type (procedure123456/abdomen+chest)

In [21]:
score_comparison 

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,...,ecs_subscale_score_Baseline,ecs_subscale_score_12mo,fact_e_total_Baseline_std,bmi,bmi_std,smoking_collapsed_imputed,alcohol_collapsed_imputed,level_tumor_collapsed_imputed,path_histology_collapsed_imputed,path_esoph_stage_collapsed_imputed
id,,,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,...,46.0,55.0,-0.485421,46.075514,3.300320,1,1,2,0,4
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,...,36.0,48.0,-1.448897,25.360019,-0.226055,1,1,2,1,2
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,...,52.0,54.0,-0.399778,34.213824,1.281119,1,0,2,1,3
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,...,37.0,57.0,-0.421189,27.115631,0.072801,0,0,1,1,3
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,...,57.0,51.0,0.178308,32.922429,1.061286,1,1,2,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,...,67.0,38.0,0.906268,27.297959,0.103839,1,1,1,1,3
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,...,58.0,35.0,1.034731,33.217993,1.111599,0,0,1,0,2
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,...,62.0,37.0,0.520877,39.100346,2.112946,1,1,1,3,3


In [23]:
def preprocess_surgery_approach(score_comparison, filtered_df):
    """
    Preprocess surgery_approach: combine abdomen_approach and chest_approach + MICE imputation for missing
    Surgery approach is categorical: 0=Open, 1=Minimally Invasive, 9999=Unknown (then imputed with MICE)
    Built from abdomen_approach and chest_approach
    
    Returns:
    - Dictionary with preprocessing results and statistics
    """
    import miceforest as mf
    
    print("="*80)
    print("STEP 2: PREDICTOR PREPROCESSING - Surgery Approach")
    print("="*80)
    print("Preparing surgery approach data for MICE imputation...")
    
    # Get all unique patient IDs from score_comparison
    all_patient_ids = score_comparison.index.tolist()
    
    # ========== Process abdomen_approach ==========
    print("\n--- Processing abdomen_approach ---")
    abdomen_prep = pd.DataFrame({'patient_id': all_patient_ids})
    abdomen_prep = abdomen_prep.merge(
        filtered_df[["id", "abdomen_approach"]],
        left_on='patient_id',
        right_on='id',
        how='left'
    )
    
    print(abdomen_prep['abdomen_approach'].value_counts())
    
    # Find patient IDs where all abdomen_approach values are missing
    patients_all_missing_abdomen = abdomen_prep.groupby('patient_id')['abdomen_approach'].apply(lambda x: x.isna().all())
    patients_to_impute_abdomen = patients_all_missing_abdomen[patients_all_missing_abdomen].index
    
    # Impute abdomen_approach for those patients with 9999 (unknown)
    mask = abdomen_prep['patient_id'].isin(patients_to_impute_abdomen)
    abdomen_prep.loc[mask, 'abdomen_approach'] = 9999.0
    print(f"Imputed {len(patients_to_impute_abdomen)} unique patients with abdomen_approach=9999 (unknown)")
    
    # Create a clean dataframe with no missing values in abdomen_approach
    abdomen_prep_clean = abdomen_prep.dropna(subset=["abdomen_approach"])
    abdomen_prep_clean = abdomen_prep_clean.drop_duplicates()
    
    # Check for duplicate patient IDs and keep first, drop last
    duplicate_ids = abdomen_prep_clean['patient_id'][abdomen_prep_clean['patient_id'].duplicated()].unique()
    if len(duplicate_ids) > 0:
        print(f"Found {len(duplicate_ids)} patients with duplicate abdomen_approach entries")
        for patient_id in duplicate_ids:
            patient_rows = abdomen_prep_clean[abdomen_prep_clean['patient_id'] == patient_id]
            last_index = patient_rows.index[-1]
            abdomen_prep_clean = abdomen_prep_clean.drop(index=last_index)
            print(f"Dropped last occurrence for patient {patient_id}")
    
    print(abdomen_prep_clean['abdomen_approach'].value_counts())
    
    # ========== Process chest_approach ==========
    print("\n--- Processing chest_approach ---")
    chest_prep = pd.DataFrame({'patient_id': all_patient_ids})
    chest_prep = chest_prep.merge(
        filtered_df[["id", "chest_approach"]],
        left_on='patient_id',
        right_on='id',
        how='left'
    )
    
    print(chest_prep['chest_approach'].value_counts())
    
    # Find patient IDs where all chest_approach values are missing
    patients_all_missing_chest = chest_prep.groupby('patient_id')['chest_approach'].apply(lambda x: x.isna().all())
    patients_to_impute_chest = patients_all_missing_chest[patients_all_missing_chest].index
    
    # Impute chest_approach for those patients with 9999 (unknown)
    mask = chest_prep['patient_id'].isin(patients_to_impute_chest)
    chest_prep.loc[mask, 'chest_approach'] = 9999.0
    print(f"Imputed {len(patients_to_impute_chest)} unique patients with chest_approach=9999 (unknown)")
    
    # Create a clean dataframe with no missing values in chest_approach
    chest_prep_clean = chest_prep.dropna(subset=["chest_approach"])
    chest_prep_clean = chest_prep_clean.drop_duplicates()
    
    # Check for duplicate patient IDs and keep first, drop last
    duplicate_ids = chest_prep_clean['patient_id'][chest_prep_clean['patient_id'].duplicated()].unique()
    if len(duplicate_ids) > 0:
        print(f"Found {len(duplicate_ids)} patients with duplicate chest_approach entries")
        for patient_id in duplicate_ids:
            patient_rows = chest_prep_clean[chest_prep_clean['patient_id'] == patient_id]
            last_index = patient_rows.index[-1]
            chest_prep_clean = chest_prep_clean.drop(index=last_index)
            print(f"Dropped last occurrence for patient {patient_id}")
    
    print(chest_prep_clean['chest_approach'].value_counts())
    
    # ========== Merge abdomen and chest approaches ==========
    print("\n--- Creating surgery approach classification ---")
    surgery_prep = abdomen_prep_clean.merge(
        chest_prep_clean[['patient_id', 'chest_approach']], 
        on='patient_id', 
        how='left'
    )
    
    surgery_prep['surgery_approach'] = 9999  # Default to missing
    
    # Define conditions for minimally invasive
    minimally_invasive_abdomen = surgery_prep['abdomen_approach'] == 2  # LAP
    minimally_invasive_chest = surgery_prep['chest_approach'].isin([2, 4])  # VATS or Prone
    
    # Define conditions for open (takes priority)
    open_abdomen = surgery_prep['abdomen_approach'].isin([1, 3])  # Open or LAP to Open
    open_chest = surgery_prep['chest_approach'].isin([1, 3])  # Open or VATS to Open
    
    # Define conditions for missing data
    missing_abdomen = surgery_prep['abdomen_approach'].isin([4, 9999]) | surgery_prep['abdomen_approach'].isna()
    missing_chest = surgery_prep['chest_approach'].isin([5, 9999]) | surgery_prep['chest_approach'].isna()
    
    # Classify surgery approach
    # Open takes priority
    surgery_prep.loc[open_abdomen | open_chest, 'surgery_approach'] = 0  # Open
    
    # Minimally invasive (only if not already classified as open)
    surgery_prep.loc[(surgery_prep['surgery_approach'] != 0) &
                        (minimally_invasive_abdomen | minimally_invasive_chest), 'surgery_approach'] = 1  # Minimally Invasive
    
    # Missing only if both approaches are missing/NA and not already classified
    surgery_prep.loc[(surgery_prep['surgery_approach'] == 9999) &
                        missing_abdomen & missing_chest, 'surgery_approach'] = 9999  # Missing
    
    print(f"Surgery approach - Value counts:\n{surgery_prep['surgery_approach'].value_counts()}")
    
    # Perform MICE imputation ONCE
    print("\nPerforming MICE imputation for surgery_approach=9999...")
    
    # Replace 9999 with NaN for imputation
    surgery_prep['surgery_approach_for_imputation'] = surgery_prep['surgery_approach'].replace(9999, np.nan)
    
    # Merge baseline FACT-E for imputation model
    surgery_prep = surgery_prep.merge(
        score_comparison[['fact_e_total_Baseline']],
        left_on='patient_id',
        right_index=True,
        how='left'
    )
    
    print(f"Running MICE on {len(surgery_prep)} unique patients")
    
    # Create dataset for MICE
    mice_data = surgery_prep[['surgery_approach_for_imputation', 'fact_e_total_Baseline']].copy()
    
    # MICE imputation
    mice_imputer = mf.ImputationKernel(
        mice_data,
        datasets=3,
        random_state=42
    )
    
    mice_imputer.mice(iterations=3, verbose=True)
    
    # Get completed dataset
    completed_data = mice_imputer.complete_data(0)
    
    # Extract imputed values
    imputed_surgery = completed_data['surgery_approach_for_imputation'].round().astype(int)
    
    # Create mapping dictionary for all patients
    surgery_imputation_map = dict(zip(surgery_prep['patient_id'], imputed_surgery))
    
    print("Surgery approach after MICE imputation - Value counts:")
    print(pd.Series(imputed_surgery).value_counts())
    print("\n" + "="*80)
    
    # Create final dataframe with imputed surgery values
    surgery_final = pd.DataFrame({
        'patient_id': list(surgery_imputation_map.keys()),
        'surgery_approach_imputed': list(surgery_imputation_map.values())
    })
    surgery_final = surgery_final.set_index('patient_id')
    
    # Calculate summary statistics
    surgery_data = surgery_final['surgery_approach_imputed']
    n_open = (surgery_data == 0).sum()
    n_minimally_invasive = (surgery_data == 1).sum()
    total = len(surgery_data)
    
    print(f"\nFinal surgery approach distribution:")
    print(f"  N: {total}")
    print(f"  Open (0): {n_open} ({n_open/total*100:.1f}%)")
    print(f"  Minimally Invasive (1): {n_minimally_invasive} ({n_minimally_invasive/total*100:.1f}%)")
    print(f"  Mean: {surgery_data.mean():.2f}")
    print(f"  SD: {surgery_data.std():.2f}")
    
    # Merge back to score_comparison
    score_comparison = score_comparison.merge(
        surgery_final[['surgery_approach_imputed']], 
        left_index=True, 
        right_index=True, 
        how='left'
    )
    
    print("\n✓ Created column: 'surgery_approach_imputed'")
    print("="*80)
    
    # Return results dictionary
    results = {
        'n_total': len(all_patient_ids),
        'n_missing_abdomen': len(patients_to_impute_abdomen),
        'n_missing_chest': len(patients_to_impute_chest),
        'n_open': n_open,
        'n_minimally_invasive': n_minimally_invasive,
        'mean': surgery_data.mean(),
        'sd': surgery_data.std(),
        'preprocessed_data': score_comparison
    }
    
    return results

# Run preprocessing
surgery_approach_summaries = preprocess_surgery_approach(score_comparison, filtered_df)

# Update score_comparison with preprocessed data
score_comparison = surgery_approach_summaries['preprocessed_data']

# Display first few rows to verify
print("\nFirst few rows of surgery_approach (imputed):")
print(score_comparison[['surgery_approach_imputed']].head(10))

STEP 2: PREDICTOR PREPROCESSING - Surgery Approach
Preparing surgery approach data for MICE imputation...

--- Processing abdomen_approach ---
abdomen_approach
2.0    39
1.0    28
3.0     2
Name: count, dtype: int64
Imputed 41 unique patients with abdomen_approach=9999 (unknown)
abdomen_approach
9999.0    41
2.0       39
1.0       26
3.0        2
Name: count, dtype: int64

--- Processing chest_approach ---
chest_approach
1.0    59
2.0    32
4.0     8
3.0     2
Name: count, dtype: int64
Imputed 12 unique patients with chest_approach=9999 (unknown)
Found 1 patients with duplicate chest_approach entries
Dropped last occurrence for patient 1049
chest_approach
1.0       55
2.0       31
9999.0    12
4.0        8
3.0        2
Name: count, dtype: int64

--- Creating surgery approach classification ---
Surgery approach - Value counts:
surgery_approach
0       61
1       40
9999     7
Name: count, dtype: int64

Performing MICE imputation for surgery_approach=9999...
Running MICE on 108 unique pa

In [24]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,...,ecs_subscale_score_12mo,fact_e_total_Baseline_std,bmi,bmi_std,smoking_collapsed_imputed,alcohol_collapsed_imputed,level_tumor_collapsed_imputed,path_histology_collapsed_imputed,path_esoph_stage_collapsed_imputed,surgery_approach_imputed
id,,,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,...,55.0,-0.485421,46.075514,3.300320,1,1,2,0,4,0
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,...,48.0,-1.448897,25.360019,-0.226055,1,1,2,1,2,1
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,...,54.0,-0.399778,34.213824,1.281119,1,0,2,1,3,0
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,...,57.0,-0.421189,27.115631,0.072801,0,0,1,1,3,0
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,...,51.0,0.178308,32.922429,1.061286,1,1,2,3,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,...,38.0,0.906268,27.297959,0.103839,1,1,1,1,3,1
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,...,35.0,1.034731,33.217993,1.111599,0,0,1,0,2,0
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,...,37.0,0.520877,39.100346,2.112946,1,1,1,3,3,0


# Characteristic by class - 8) Neoadjuvant treatment

In [27]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,...,ecs_subscale_score_12mo,fact_e_total_Baseline_std,bmi,bmi_std,smoking_collapsed_imputed,alcohol_collapsed_imputed,level_tumor_collapsed_imputed,path_histology_collapsed_imputed,path_esoph_stage_collapsed_imputed,surgery_approach_imputed
id,,,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,...,55.0,-0.485421,46.075514,3.300320,1,1,2,0,4,0
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,...,48.0,-1.448897,25.360019,-0.226055,1,1,2,1,2,1
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,...,54.0,-0.399778,34.213824,1.281119,1,0,2,1,3,0
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,...,57.0,-0.421189,27.115631,0.072801,0,0,1,1,3,0
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,...,51.0,0.178308,32.922429,1.061286,1,1,2,3,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,...,38.0,0.906268,27.297959,0.103839,1,1,1,1,3,1
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,...,35.0,1.034731,33.217993,1.111599,0,0,1,0,2,0
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,...,37.0,0.520877,39.100346,2.112946,1,1,1,3,3,0


In [28]:
def preprocess_neotx(score_comparison, filtered_df):
    """
    Preprocess neotx: collapse categories + MICE imputation for missing
    Neotx is categorical: 0=None (1), 1=Chemotherapy (2), 2=Chemoradiotherapy (4), 3=Other (3,5,6,7), 9999=Unknown (then imputed with MICE)
    
    Returns:
    - Dictionary with preprocessing results and statistics
    """
    import miceforest as mf
    
    print("="*80)
    print("STEP 2: PREDICTOR PREPROCESSING - Neoadjuvant Treatment")
    print("="*80)
    print("Preparing neoadjuvant treatment data for MICE imputation...")
    
    # Get all unique patient IDs from score_comparison
    all_patient_ids = score_comparison.index.tolist()
    
    # Merge with filtered_df to get neotx columns
    neotx_prep = pd.DataFrame({'patient_id': all_patient_ids})
    neotx_prep = neotx_prep.merge(
        filtered_df[["id", "neotx___notx", "neotx___chemo", "neotx___rads", 
                        "neotx___chemorads", "neotx___immuno", "neotx___other"]],
        left_on='patient_id',
        right_on='id',
        how='left'
    )
    
    # Create neotx mapping
    neotx_mapping = {
        "neotx___notx": 1,
        "neotx___chemo": 2,
        "neotx___rads": 3,
        "neotx___chemorads": 4,
        "neotx___immuno": 5,
        "neotx___other": 6
    }
    
    # Assign neotx
    def assign_neotx(row):
        active_cols = [col for col in neotx_mapping.keys() if row[col] == 1]
        if len(active_cols) == 1:
            return neotx_mapping[active_cols[0]]
        elif len(active_cols) > 1:
            return 7  # mixed
        else:
            return None  # missing/NaN
    
    neotx_prep['neotx'] = neotx_prep.apply(assign_neotx, axis=1)
    
    # Group by patient_id and check if patient has multiple different neotx types
    patient_neotx = neotx_prep.groupby('patient_id')['neotx'].apply(
        lambda x: 7 if len(set(x.dropna())) > 1 else x.dropna().iloc[0] if len(x.dropna()) > 0 else None
    ).reset_index()
    patient_neotx.columns = ['patient_id', 'final_neotx']
    
    # Merge back
    neotx_prep = neotx_prep.merge(patient_neotx[['patient_id', 'final_neotx']], on='patient_id', how='left')
    
    # Drop the individual neotx columns and rename final_neotx
    neotx_prep = neotx_prep.drop(columns=[
        "neotx___notx",
        "neotx___chemo",
        "neotx___rads",
        "neotx___chemorads",
        "neotx___immuno",
        "neotx___other",
        "neotx"
    ]).rename(columns={"final_neotx": "neotx"})
    
    print(neotx_prep['neotx'].value_counts())
    
    # Find patient IDs where all neotx values are missing
    patients_all_missing = neotx_prep.groupby('patient_id')['neotx'].apply(lambda x: x.isna().all())
    patients_to_impute = patients_all_missing[patients_all_missing].index
    
    # Impute neotx for those patients with 9999 (unknown)
    mask = neotx_prep['patient_id'].isin(patients_to_impute)
    neotx_prep.loc[mask, 'neotx'] = 9999.0
    print(f"Imputed {len(patients_to_impute)} unique patients with neotx=9999 (unknown)")
    
    # Create a clean dataframe with no missing values in neotx
    neotx_prep_clean = neotx_prep.dropna(subset=["neotx"])
    neotx_prep_clean = neotx_prep_clean.drop_duplicates()
    
    print(neotx_prep_clean['neotx'].value_counts())
    
    # Collapse neotx variable
    print("\nCollapsing neotx variable...")
    neotx_prep_clean['neotx_collapsed'] = neotx_prep_clean['neotx'].copy()
    neotx_prep_clean.loc[neotx_prep_clean['neotx'] == 1, 'neotx_collapsed'] = 0  # None
    neotx_prep_clean.loc[neotx_prep_clean['neotx'] == 2, 'neotx_collapsed'] = 1  # Chemotherapy
    neotx_prep_clean.loc[neotx_prep_clean['neotx'] == 4, 'neotx_collapsed'] = 2  # Chemoradiotherapy
    neotx_prep_clean.loc[neotx_prep_clean['neotx'].isin([3, 5, 6, 7]), 'neotx_collapsed'] = 3  # Other
    neotx_prep_clean.loc[neotx_prep_clean['neotx'] == 9999, 'neotx_collapsed'] = 9999  # Unknown
    
    print("Neotx collapsed - Value counts:")
    print(neotx_prep_clean['neotx_collapsed'].value_counts())
    
    # Perform MICE imputation ONCE
    print("\nPerforming MICE imputation for neotx=9999...")
    
    # Replace 9999 with NaN for imputation
    neotx_prep_clean['neotx_collapsed_for_imputation'] = neotx_prep_clean['neotx_collapsed'].replace(9999, np.nan)
    
    # Merge baseline FACT-E for imputation model
    neotx_prep_clean = neotx_prep_clean.merge(
        score_comparison[['fact_e_total_Baseline']],
        left_on='patient_id',
        right_index=True,
        how='left'
    )
    
    print(f"Running MICE on {len(neotx_prep_clean)} unique patients")
    
    # Create dataset for MICE
    mice_data = neotx_prep_clean[['neotx_collapsed_for_imputation', 'fact_e_total_Baseline']].copy()
    
    # MICE imputation
    mice_imputer = mf.ImputationKernel(
        mice_data,
        datasets=3,
        random_state=42
    )
    
    mice_imputer.mice(iterations=3, verbose=True)
    
    # Get completed dataset
    completed_data = mice_imputer.complete_data(0)
    
    # Extract imputed values
    imputed_neotx = completed_data['neotx_collapsed_for_imputation'].round().astype(int)
    
    # Create mapping dictionary for all patients
    neotx_imputation_map = dict(zip(neotx_prep_clean['patient_id'], imputed_neotx))
    
    print("Neotx after MICE imputation - Value counts:")
    print(pd.Series(imputed_neotx).value_counts())
    print("\n" + "="*80)
    
    # Create final dataframe with imputed neotx values
    neotx_final = pd.DataFrame({
        'patient_id': list(neotx_imputation_map.keys()),
        'neotx_collapsed_imputed': list(neotx_imputation_map.values())
    })
    neotx_final = neotx_final.set_index('patient_id')
    
    # Calculate summary statistics
    neotx_data = neotx_final['neotx_collapsed_imputed']
    n_none = (neotx_data == 0).sum()
    n_chemo = (neotx_data == 1).sum()
    n_chemorads = (neotx_data == 2).sum()
    n_other = (neotx_data == 3).sum()
    total = len(neotx_data)
    
    print(f"\nFinal neoadjuvant treatment distribution:")
    print(f"  N: {total}")
    print(f"  None (0): {n_none} ({n_none/total*100:.1f}%)")
    print(f"  Chemotherapy (1): {n_chemo} ({n_chemo/total*100:.1f}%)")
    print(f"  Chemoradiotherapy (2): {n_chemorads} ({n_chemorads/total*100:.1f}%)")
    print(f"  Other (3): {n_other} ({n_other/total*100:.1f}%)")
    print(f"  Mean: {neotx_data.mean():.2f}")
    print(f"  SD: {neotx_data.std():.2f}")
    
    # Merge back to score_comparison
    score_comparison = score_comparison.merge(
        neotx_final[['neotx_collapsed_imputed']], 
        left_index=True, 
        right_index=True, 
        how='left'
    )
    
    print("\n✓ Created column: 'neotx_collapsed_imputed'")
    print("="*80)
    
    # Return results dictionary
    results = {
        'n_total': len(all_patient_ids),
        'n_missing': len(patients_to_impute),
        'n_none': n_none,
        'n_chemo': n_chemo,
        'n_chemorads': n_chemorads,
        'n_other': n_other,
        'mean': neotx_data.mean(),
        'sd': neotx_data.std(),
        'preprocessed_data': score_comparison
    }
    
    return results

# Run preprocessing
neotx_summaries = preprocess_neotx(score_comparison, filtered_df)

# Update score_comparison with preprocessed data
score_comparison = neotx_summaries['preprocessed_data']

# Display first few rows to verify
print("\nFirst few rows of neotx (imputed):")
print(score_comparison[['neotx_collapsed_imputed']].head(10))

STEP 2: PREDICTOR PREPROCESSING - Neoadjuvant Treatment
Preparing neoadjuvant treatment data for MICE imputation...
neotx
2.0    946
7.0    238
4.0    130
1.0    127
6.0     22
3.0     12
Name: count, dtype: int64
Imputed 3 unique patients with neotx=9999 (unknown)
neotx
2.0       66
7.0       17
1.0       10
4.0        9
9999.0     3
6.0        2
3.0        1
Name: count, dtype: int64

Collapsing neotx variable...
Neotx collapsed - Value counts:
neotx_collapsed
1.0       66
3.0       20
0.0       10
2.0        9
9999.0     3
Name: count, dtype: int64

Performing MICE imputation for neotx=9999...
Running MICE on 108 unique patients
Initialized logger with name mice 1-3
Dataset 0
1  | neotx_collapsed_for_imputation
2  | neotx_collapsed_for_imputation
3  | neotx_collapsed_for_imputation
Dataset 1
1  | neotx_collapsed_for_imputation
2  | neotx_collapsed_for_imputation
3  | neotx_collapsed_for_imputation
Dataset 2
1  | neotx_collapsed_for_imputation
2  | neotx_collapsed_for_imputation
3  |

In [29]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,...,fact_e_total_Baseline_std,bmi,bmi_std,smoking_collapsed_imputed,alcohol_collapsed_imputed,level_tumor_collapsed_imputed,path_histology_collapsed_imputed,path_esoph_stage_collapsed_imputed,surgery_approach_imputed,neotx_collapsed_imputed
id,,,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,...,-0.485421,46.075514,3.300320,1,1,2,0,4,0,1
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,...,-1.448897,25.360019,-0.226055,1,1,2,1,2,1,1
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,...,-0.399778,34.213824,1.281119,1,0,2,1,3,0,1
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,...,-0.421189,27.115631,0.072801,0,0,1,1,3,0,1
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,...,0.178308,32.922429,1.061286,1,1,2,3,2,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,...,0.906268,27.297959,0.103839,1,1,1,1,3,1,1
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,...,1.034731,33.217993,1.111599,0,0,1,0,2,0,1
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,...,0.520877,39.100346,2.112946,1,1,1,3,3,0,1


# Characteristic by class - 9) age (age_diagnosis)

In [31]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,...,fact_e_total_Baseline_std,bmi,bmi_std,smoking_collapsed_imputed,alcohol_collapsed_imputed,level_tumor_collapsed_imputed,path_histology_collapsed_imputed,path_esoph_stage_collapsed_imputed,surgery_approach_imputed,neotx_collapsed_imputed
id,,,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,...,-0.485421,46.075514,3.300320,1,1,2,0,4,0,1
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,...,-1.448897,25.360019,-0.226055,1,1,2,1,2,1,1
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,...,-0.399778,34.213824,1.281119,1,0,2,1,3,0,1
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,...,-0.421189,27.115631,0.072801,0,0,1,1,3,0,1
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,...,0.178308,32.922429,1.061286,1,1,2,3,2,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,...,0.906268,27.297959,0.103839,1,1,1,1,3,1,1
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,...,1.034731,33.217993,1.111599,0,0,1,0,2,0,1
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,...,0.520877,39.100346,2.112946,1,1,1,3,3,0,1


In [32]:
def preprocess_age(score_comparison, filtered_df):
    """
    Preprocess age: median imputation for missing + z-score standardization
    Age is continuous variable, missing values imputed with median
    
    Returns:
    - Dictionary with preprocessing results and statistics
    """
    
    print("="*80)
    print("STEP 2: PREDICTOR PREPROCESSING - Age at Diagnosis")
    print("="*80)
    print("Preparing age data...")
    
    # Get all unique patient IDs from score_comparison
    all_patient_ids = score_comparison.index.tolist()
    
    # Merge with filtered_df to get age_diagnosis
    age_prep = pd.DataFrame({'patient_id': all_patient_ids})
    age_prep = age_prep.merge(
        filtered_df[["id", "age_diagnosis"]],
        left_on='patient_id',
        right_on='id',
        how='left'
    )
    
    # Calculate median age from all non-missing values
    median_age = age_prep['age_diagnosis'].median()
    
    # Find patient IDs where all age values are missing
    patients_all_missing = age_prep.groupby('patient_id')['age_diagnosis'].apply(lambda x: x.isna().all())
    patients_to_impute = patients_all_missing[patients_all_missing].index
    
    # Impute age for those patients
    mask = age_prep['patient_id'].isin(patients_to_impute)
    age_prep.loc[mask, 'age_diagnosis'] = median_age
    print(f"Imputed {len(patients_to_impute)} unique patients with median age: {median_age:.2f}")
    
    # Create a clean dataframe with no missing values in age
    age_prep_clean = age_prep.dropna(subset=["age_diagnosis"])
    age_prep_clean = age_prep_clean.drop_duplicates()
    
    print(f"Age range: {age_prep_clean['age_diagnosis'].min():.1f} - {age_prep_clean['age_diagnosis'].max():.1f}")
    print(f"Age mean: {age_prep_clean['age_diagnosis'].mean():.1f} (SD: {age_prep_clean['age_diagnosis'].std():.1f})")
    
    # Calculate mean and SD for z-score standardization
    mean_age = age_prep_clean['age_diagnosis'].mean()
    sd_age = age_prep_clean['age_diagnosis'].std()
    
    print(f"\nAge statistics after imputation:")
    print(f"  Mean: {mean_age:.2f}")
    print(f"  SD: {sd_age:.2f}")
    print(f"  Min: {age_prep_clean['age_diagnosis'].min():.2f}")
    print(f"  Max: {age_prep_clean['age_diagnosis'].max():.2f}")
    
    # Z-score standardization
    age_prep_clean['age_diagnosis_std'] = (age_prep_clean['age_diagnosis'] - mean_age) / sd_age
    
    print(f"\nStandardized age statistics:")
    print(f"  Mean: {age_prep_clean['age_diagnosis_std'].mean():.2f}")
    print(f"  SD: {age_prep_clean['age_diagnosis_std'].std():.2f}")
    print(f"  Min: {age_prep_clean['age_diagnosis_std'].min():.2f}")
    print(f"  Max: {age_prep_clean['age_diagnosis_std'].max():.2f}")
    
    # Create final dataframe
    age_final = age_prep_clean[['patient_id', 'age_diagnosis', 'age_diagnosis_std']].drop_duplicates(subset=['patient_id'])
    age_final = age_final.set_index('patient_id')
    
    # Merge back to score_comparison
    score_comparison = score_comparison.merge(
        age_final[['age_diagnosis', 'age_diagnosis_std']], 
        left_index=True, 
        right_index=True, 
        how='left'
    )
    
    print("\n✓ Created columns: 'age_diagnosis' and 'age_diagnosis_std'")
    print("="*80)
    
    # Return results dictionary
    results = {
        'n_total': len(all_patient_ids),
        'n_missing': len(patients_to_impute),
        'median': median_age,
        'mean': mean_age,
        'sd': sd_age,
        'min': age_prep_clean['age_diagnosis'].min(),
        'max': age_prep_clean['age_diagnosis'].max(),
        'preprocessed_data': score_comparison
    }
    
    return results

# Run preprocessing
age_summaries = preprocess_age(score_comparison, filtered_df)

# Update score_comparison with preprocessed data
score_comparison = age_summaries['preprocessed_data']

# Display first few rows to verify
print("\nFirst few rows of age (original and standardized):")
print(score_comparison[['age_diagnosis', 'age_diagnosis_std']].head(10))

STEP 2: PREDICTOR PREPROCESSING - Age at Diagnosis
Preparing age data...
Imputed 11 unique patients with median age: 65.48
Age range: 29.7 - 83.6
Age mean: 64.3 (SD: 8.8)

Age statistics after imputation:
  Mean: 64.29
  SD: 8.81
  Min: 29.72
  Max: 83.59

Standardized age statistics:
  Mean: -0.00
  SD: 1.00
  Min: -3.92
  Max: 2.19

✓ Created columns: 'age_diagnosis' and 'age_diagnosis_std'

First few rows of age (original and standardized):
     age_diagnosis  age_diagnosis_std
id                                   
32       29.722718          -3.924010
76       57.597350          -0.759442
125      64.110830          -0.019975
160      50.804602          -1.530613
198      66.229970           0.220608
230      65.780954           0.169632
275      65.479727           0.135434
307      65.479727           0.135434
364      50.117388          -1.608632
442      66.098437           0.205675


In [33]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,...,bmi_std,smoking_collapsed_imputed,alcohol_collapsed_imputed,level_tumor_collapsed_imputed,path_histology_collapsed_imputed,path_esoph_stage_collapsed_imputed,surgery_approach_imputed,neotx_collapsed_imputed,age_diagnosis,age_diagnosis_std
id,,,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,...,3.300320,1,1,2,0,4,0,1,29.722718,-3.924010
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,...,-0.226055,1,1,2,1,2,1,1,57.597350,-0.759442
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,...,1.281119,1,0,2,1,3,0,1,64.110830,-0.019975
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,...,0.072801,0,0,1,1,3,0,1,50.804602,-1.530613
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,...,1.061286,1,1,2,3,2,1,1,66.229970,0.220608
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,...,0.103839,1,1,1,1,3,1,1,69.876863,0.634635
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,...,1.111599,0,0,1,0,2,0,1,79.245980,1.698298
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,...,2.112946,1,1,1,3,3,0,1,60.529648,-0.426542


# Characteristic by class - 10) sex

In [34]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,...,bmi_std,smoking_collapsed_imputed,alcohol_collapsed_imputed,level_tumor_collapsed_imputed,path_histology_collapsed_imputed,path_esoph_stage_collapsed_imputed,surgery_approach_imputed,neotx_collapsed_imputed,age_diagnosis,age_diagnosis_std
id,,,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,...,3.300320,1,1,2,0,4,0,1,29.722718,-3.924010
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,...,-0.226055,1,1,2,1,2,1,1,57.597350,-0.759442
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,...,1.281119,1,0,2,1,3,0,1,64.110830,-0.019975
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,...,0.072801,0,0,1,1,3,0,1,50.804602,-1.530613
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,...,1.061286,1,1,2,3,2,1,1,66.229970,0.220608
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,...,0.103839,1,1,1,1,3,1,1,69.876863,0.634635
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,...,1.111599,0,0,1,0,2,0,1,79.245980,1.698298
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,...,2.112946,1,1,1,3,3,0,1,60.529648,-0.426542


In [35]:
def preprocess_gender(score_comparison, filtered_df):
    """
    Preprocess gender: handle missing values (no MICE needed if no missing)
    Gender is categorical: 1=Male, 2=Female, 9999=Unknown (if any missing)
    
    Returns:
    - Dictionary with preprocessing results and statistics
    """
    
    print("="*80)
    print("STEP 2: PREDICTOR PREPROCESSING - Gender")
    print("="*80)
    print("Preparing gender data...")
    
    # Get all unique patient IDs from score_comparison
    all_patient_ids = score_comparison.index.tolist()
    
    # Merge with filtered_df to get gender
    gender_prep = pd.DataFrame({'patient_id': all_patient_ids})
    gender_prep = gender_prep.merge(
        filtered_df[["id", "gender"]],
        left_on='patient_id',
        right_on='id',
        how='left'
    )
    
    print(gender_prep['gender'].value_counts())
    
    # Find patient IDs where all gender values are missing
    patients_all_missing = gender_prep.groupby('patient_id')['gender'].apply(lambda x: x.isna().all())
    patients_to_impute = patients_all_missing[patients_all_missing].index
    
    # Impute gender for those patients with 9999 (unknown)
    mask = gender_prep['patient_id'].isin(patients_to_impute)
    gender_prep.loc[mask, 'gender'] = 9999.0
    print(f"Imputed {len(patients_to_impute)} unique patients with gender=9999 (unknown)")
    
    # Create a clean dataframe with no missing values in gender
    gender_prep_clean = gender_prep.dropna(subset=["gender"])
    gender_prep_clean = gender_prep_clean.drop_duplicates()
    
    print(gender_prep_clean['gender'].value_counts())
    
    # Create final dataframe
    gender_final = gender_prep_clean[['patient_id', 'gender']].drop_duplicates(subset=['patient_id'])
    gender_final = gender_final.set_index('patient_id')
    
    # Calculate summary statistics
    gender_data = gender_final['gender']
    n_male = (gender_data == 1).sum()
    n_female = (gender_data == 2).sum()
    n_unknown = (gender_data == 9999).sum()
    total = len(gender_data)
    
    print(f"\nFinal gender distribution:")
    print(f"  N: {total}")
    print(f"  Male (1): {n_male} ({n_male/total*100:.1f}%)")
    print(f"  Female (2): {n_female} ({n_female/total*100:.1f}%)")
    print(f"  Unknown (9999): {n_unknown} ({n_unknown/total*100:.1f}%)")
    print(f"  Mean: {gender_data.mean():.2f}")
    print(f"  SD: {gender_data.std():.2f}")
    
    # Merge back to score_comparison
    score_comparison = score_comparison.merge(
        gender_final[['gender']], 
        left_index=True, 
        right_index=True, 
        how='left'
    )
    
    print("\n✓ Created column: 'gender'")
    print("="*80)
    
    # Return results dictionary
    results = {
        'n_total': len(all_patient_ids),
        'n_missing': len(patients_to_impute),
        'n_male': n_male,
        'n_female': n_female,
        'n_unknown': n_unknown,
        'mean': gender_data.mean(),
        'sd': gender_data.std(),
        'preprocessed_data': score_comparison
    }
    
    return results

# Run preprocessing
gender_summaries = preprocess_gender(score_comparison, filtered_df)

# Update score_comparison with preprocessed data
score_comparison = gender_summaries['preprocessed_data']

# Display first few rows to verify
print("\nFirst few rows of gender:")
print(score_comparison[['gender']].head(10))

STEP 2: PREDICTOR PREPROCESSING - Gender
Preparing gender data...
gender
1.0    86
2.0    25
Name: count, dtype: int64
Imputed 0 unique patients with gender=9999 (unknown)
gender
1.0    84
2.0    24
Name: count, dtype: int64

Final gender distribution:
  N: 108
  Male (1): 84 (77.8%)
  Female (2): 24 (22.2%)
  Unknown (9999): 0 (0.0%)
  Mean: 1.22
  SD: 0.42

✓ Created column: 'gender'

First few rows of gender:
     gender
id         
32      1.0
76      1.0
125     1.0
160     1.0
198     2.0
230     2.0
275     1.0
307     1.0
364     2.0
442     2.0


In [36]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,...,smoking_collapsed_imputed,alcohol_collapsed_imputed,level_tumor_collapsed_imputed,path_histology_collapsed_imputed,path_esoph_stage_collapsed_imputed,surgery_approach_imputed,neotx_collapsed_imputed,age_diagnosis,age_diagnosis_std,gender
id,,,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,...,1,1,2,0,4,0,1,29.722718,-3.924010,1.0
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,...,1,1,2,1,2,1,1,57.597350,-0.759442,1.0
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,...,1,0,2,1,3,0,1,64.110830,-0.019975,1.0
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,...,0,0,1,1,3,0,1,50.804602,-1.530613,1.0
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,...,1,1,2,3,2,1,1,66.229970,0.220608,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,...,1,1,1,1,3,1,1,69.876863,0.634635,1.0
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,...,0,0,1,0,2,0,1,79.245980,1.698298,1.0
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,...,1,1,1,3,3,0,1,60.529648,-0.426542,1.0


# Characteristic by class - 11) charlson comorbidity index (cci_total)

In [38]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,...,smoking_collapsed_imputed,alcohol_collapsed_imputed,level_tumor_collapsed_imputed,path_histology_collapsed_imputed,path_esoph_stage_collapsed_imputed,surgery_approach_imputed,neotx_collapsed_imputed,age_diagnosis,age_diagnosis_std,gender
id,,,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,...,1,1,2,0,4,0,1,29.722718,-3.924010,1.0
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,...,1,1,2,1,2,1,1,57.597350,-0.759442,1.0
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,...,1,0,2,1,3,0,1,64.110830,-0.019975,1.0
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,...,0,0,1,1,3,0,1,50.804602,-1.530613,1.0
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,...,1,1,2,3,2,1,1,66.229970,0.220608,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,...,1,1,1,1,3,1,1,69.876863,0.634635,1.0
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,...,0,0,1,0,2,0,1,79.245980,1.698298,1.0
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,...,1,1,1,3,3,0,1,60.529648,-0.426542,1.0


In [39]:
def preprocess_cci(score_comparison, filtered_df):
    """
    Preprocess CCI: median imputation for missing + z-score standardization
    CCI is continuous variable, missing values imputed with median
    Also creates categorical version: 0=None (0), 1=Mild (1-2), 2=Moderate (3-4), 3=Severe (5+)
    
    Returns:
    - Dictionary with preprocessing results and statistics
    """
    
    print("="*80)
    print("STEP 2: PREDICTOR PREPROCESSING - Charlson Comorbidity Index")
    print("="*80)
    print("Preparing Charlson Comorbidity Index data...")
    
    # Get all unique patient IDs from score_comparison
    all_patient_ids = score_comparison.index.tolist()
    
    # Merge with filtered_df to get cci_total
    cci_prep = pd.DataFrame({'patient_id': all_patient_ids})
    cci_prep = cci_prep.merge(
        filtered_df[["id", "cci_total"]],
        left_on='patient_id',
        right_on='id',
        how='left'
    )
    
    # Calculate median CCI from all non-missing values
    median_cci = cci_prep['cci_total'].median()
    
    # Find patient IDs where all CCI values are missing
    patients_all_missing = cci_prep.groupby('patient_id')['cci_total'].apply(lambda x: x.isna().all())
    patients_to_impute = patients_all_missing[patients_all_missing].index
    
    # Impute CCI for those patients
    mask = cci_prep['patient_id'].isin(patients_to_impute)
    cci_prep.loc[mask, 'cci_total'] = median_cci
    print(f"Imputed {len(patients_to_impute)} unique patients with median CCI: {median_cci:.2f}")
    
    # Create a clean dataframe with no missing values in CCI
    cci_prep_clean = cci_prep.dropna(subset=["cci_total"])
    cci_prep_clean = cci_prep_clean.drop_duplicates()
    
    # Define the categorization function
    def categorize_cci(cci):
        if cci == 0:
            return 0  # "None"
        elif cci <= 2:
            return 1  # "Mild"
        elif cci <= 4:
            return 2  # "Moderate"
        else:
            return 3  # "Severe"
    
    # Apply the categorization function to create the categorical variable
    cci_prep_clean['cci_category_numeric'] = cci_prep_clean['cci_total'].apply(categorize_cci)
    
    print(f"\nCCI range: {cci_prep_clean['cci_total'].min():.1f} - {cci_prep_clean['cci_total'].max():.1f}")
    print(f"CCI mean: {cci_prep_clean['cci_total'].mean():.1f} (SD: {cci_prep_clean['cci_total'].std():.1f})")
    print("\nCCI categorical distribution:")
    print(cci_prep_clean['cci_category_numeric'].value_counts().sort_index())
    
    # Calculate mean and SD for z-score standardization
    mean_cci = cci_prep_clean['cci_total'].mean()
    sd_cci = cci_prep_clean['cci_total'].std()
    
    print(f"\nCCI statistics after imputation:")
    print(f"  Mean: {mean_cci:.2f}")
    print(f"  SD: {sd_cci:.2f}")
    print(f"  Min: {cci_prep_clean['cci_total'].min():.2f}")
    print(f"  Max: {cci_prep_clean['cci_total'].max():.2f}")
    
    # Z-score standardization
    cci_prep_clean['cci_total_std'] = (cci_prep_clean['cci_total'] - mean_cci) / sd_cci
    
    print(f"\nStandardized CCI statistics:")
    print(f"  Mean: {cci_prep_clean['cci_total_std'].mean():.2f}")
    print(f"  SD: {cci_prep_clean['cci_total_std'].std():.2f}")
    print(f"  Min: {cci_prep_clean['cci_total_std'].min():.2f}")
    print(f"  Max: {cci_prep_clean['cci_total_std'].max():.2f}")
    
    # Create final dataframe
    cci_final = cci_prep_clean[['patient_id', 'cci_total', 'cci_total_std', 'cci_category_numeric']].drop_duplicates(subset=['patient_id'])
    cci_final = cci_final.set_index('patient_id')
    
    # Calculate categorical summary statistics
    cci_cat_data = cci_final['cci_category_numeric']
    n_none = (cci_cat_data == 0).sum()
    n_mild = (cci_cat_data == 1).sum()
    n_moderate = (cci_cat_data == 2).sum()
    n_severe = (cci_cat_data == 3).sum()
    total = len(cci_cat_data)
    
    print(f"\nFinal CCI categorical distribution:")
    print(f"  None (0): {n_none} ({n_none/total*100:.1f}%)")
    print(f"  Mild (1): {n_mild} ({n_mild/total*100:.1f}%)")
    print(f"  Moderate (2): {n_moderate} ({n_moderate/total*100:.1f}%)")
    print(f"  Severe (3): {n_severe} ({n_severe/total*100:.1f}%)")
    
    # Merge back to score_comparison
    score_comparison = score_comparison.merge(
        cci_final[['cci_total', 'cci_total_std', 'cci_category_numeric']], 
        left_index=True, 
        right_index=True, 
        how='left'
    )
    
    print("\n✓ Created columns: 'cci_total', 'cci_total_std', 'cci_category_numeric'")
    print("="*80)
    
    # Return results dictionary
    results = {
        'n_total': len(all_patient_ids),
        'n_missing': len(patients_to_impute),
        'median': median_cci,
        'mean': mean_cci,
        'sd': sd_cci,
        'min': cci_prep_clean['cci_total'].min(),
        'max': cci_prep_clean['cci_total'].max(),
        'n_none': n_none,
        'n_mild': n_mild,
        'n_moderate': n_moderate,
        'n_severe': n_severe,
        'preprocessed_data': score_comparison
    }
    
    return results

# Run preprocessing
cci_summaries = preprocess_cci(score_comparison, filtered_df)

# Update score_comparison with preprocessed data
score_comparison = cci_summaries['preprocessed_data']

# Display first few rows to verify
print("\nFirst few rows of CCI (original, standardized, and categorical):")
print(score_comparison[['cci_total', 'cci_total_std', 'cci_category_numeric']].head(10))

STEP 2: PREDICTOR PREPROCESSING - Charlson Comorbidity Index
Preparing Charlson Comorbidity Index data...
Imputed 22 unique patients with median CCI: 4.00

CCI range: 0.0 - 9.0
CCI mean: 3.9 (SD: 1.5)

CCI categorical distribution:
cci_category_numeric
0     1
1    19
2    59
3    29
Name: count, dtype: int64

CCI statistics after imputation:
  Mean: 3.94
  SD: 1.46
  Min: 0.00
  Max: 9.00

Standardized CCI statistics:
  Mean: 0.00
  SD: 1.00
  Min: -2.69
  Max: 3.47

Final CCI categorical distribution:
  None (0): 1 (0.9%)
  Mild (1): 19 (17.6%)
  Moderate (2): 59 (54.6%)
  Severe (3): 29 (26.9%)

✓ Created columns: 'cci_total', 'cci_total_std', 'cci_category_numeric'

First few rows of CCI (original, standardized, and categorical):
     cci_total  cci_total_std  cci_category_numeric
id                                                 
32         2.0      -1.324120                     1
76         4.0       0.044349                     2
125        4.0       0.044349                   

In [40]:
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,...,path_histology_collapsed_imputed,path_esoph_stage_collapsed_imputed,surgery_approach_imputed,neotx_collapsed_imputed,age_diagnosis,age_diagnosis_std,gender,cci_total,cci_total_std,cci_category_numeric
id,,,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,...,0,4,0,1,29.722718,-3.924010,1.0,2.0,-1.324120,1
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,...,1,2,1,1,57.597350,-0.759442,1.0,4.0,0.044349,2
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,...,1,3,0,1,64.110830,-0.019975,1.0,4.0,0.044349,2
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,...,1,3,0,1,50.804602,-1.530613,1.0,4.0,0.044349,2
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,...,3,2,1,1,66.229970,0.220608,2.0,4.0,0.044349,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,...,1,3,1,1,69.876863,0.634635,1.0,4.0,0.044349,2
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,...,0,2,0,1,79.245980,1.698298,1.0,6.0,1.412817,3
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,...,3,3,0,1,60.529648,-0.426542,1.0,4.0,0.044349,2


# variable selection by LASSO

In [4]:
score_comparison=pd.read_csv("score_comparison_preprocessed.csv", index_col='id')
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,...,path_histology_collapsed_imputed,path_esoph_stage_collapsed_imputed,surgery_approach_imputed,neotx_collapsed_imputed,age_diagnosis,age_diagnosis_std,gender,cci_total,cci_total_std,cci_category_numeric
id,,,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,...,0,4,0,1,29.722718,-3.924010,1.0,2.0,-1.324120,1
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,...,1,2,1,1,57.597350,-0.759442,1.0,4.0,0.044349,2
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,...,1,3,0,1,64.110830,-0.019975,1.0,4.0,0.044349,2
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,...,1,3,0,1,50.804602,-1.530613,1.0,4.0,0.044349,2
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,...,3,2,1,1,66.229970,0.220608,2.0,4.0,0.044349,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,...,1,3,1,1,69.876863,0.634635,1.0,4.0,0.044349,2
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,...,0,2,0,1,79.245980,1.698298,1.0,6.0,1.412817,3
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,...,3,3,0,1,60.529648,-0.426542,1.0,4.0,0.044349,2


In [5]:
def perform_nested_lasso_cv_with_bootstrap(score_comparison):
    """
    Perform nested 10-fold CV LASSO regression with 500 bootstrap iterations
    to identify robust and stable predictive baseline variables for each of the six FACT-E domains.
    
    CORRECTED WORKFLOW:
    1. Bootstrap (500 iterations) → Identify stable features (≥50% selection)
    2. Nested CV (using ONLY stable features) → Get unbiased performance estimates
    3. Fit final LASSO on full data (using ONLY stable features) → For coefficient reference
    
    The stable features will be passed to Step 4 for final OLS regression.
    
    Returns:
    - Dictionary with results for each domain including:
        - Stable features list (to be used in Step 4)
        - Nested CV performance (R², RMSE) using stable features
        - Bootstrap selection frequencies
        - Final LASSO coefficients (using stable features)
    """
    from sklearn.linear_model import LassoCV, Lasso
    from sklearn.model_selection import KFold
    from sklearn.metrics import mean_squared_error, r2_score
    from sklearn.utils import resample
    import warnings
    warnings.filterwarnings('ignore')
    
    print("="*80)
    print("STEP 3: NESTED LASSO REGRESSION WITH BOOTSTRAP STABILITY SELECTION")
    print("="*80)
    
    # Define the six FACT-E outcome domains (12-month outcomes)
    outcome_domains = [
        'fact_e_total_12mo',
        'pwb_subscale_score_12mo',
        'swb_subscale_score_12mo',
        'ewb_subscale_score_12mo',
        'fwb_subscale_score_12mo',
        'ecs_subscale_score_12mo'
    ]
    
    # Define predictor variables (12 baseline predictors)
    predictor_cols = [
        'fact_e_total_Baseline_std',    # Continuous (standardized)
        'bmi_std',                       # Continuous (standardized)
        'age_diagnosis_std',             # Continuous (standardized)
        'cci_category_numeric',          # Categorical (0=None, 1=Mild, 2=Moderate, 3=Severe)
        'gender',                        # Categorical (1=Male, 2=Female, 9999=Unknown)
        'smoking_collapsed_imputed',     # Categorical (0=Never, 1=Ever)
        'alcohol_collapsed_imputed',     # Categorical (0=Non-drinker, 1=Ever drank)
        'level_tumor_collapsed_imputed', # Categorical (1=Cardia/GEJ, 2=Lower, 3=Middle/Upper)
        'path_histology_collapsed_imputed', # Categorical (0=No carcinoma, 1=Adeno, 2=SCC, 3=Other)
        'path_esoph_stage_collapsed_imputed', # Categorical (0-5)
        'surgery_approach_imputed',      # Categorical (0=Open, 1=Minimally Invasive)
        'neotx_collapsed_imputed'        # Categorical (0=None, 1=Chemo, 2=Chemorads, 3=Other)
    ]
    
    # Check if all predictor columns exist
    missing_cols = [col for col in predictor_cols if col not in score_comparison.columns]
    if missing_cols:
        print(f"ERROR: Missing predictor columns: {missing_cols}")
        return None
    
    # Prepare predictor data
    X = score_comparison[predictor_cols].copy()
    
    # Check for any remaining missing values
    n_missing = X.isnull().sum().sum()
    if n_missing > 0:
        print(f"WARNING: Found {n_missing} missing values in predictors")
        print(X.isnull().sum())
        return None
    
    print(f"\nTotal patients: {len(X)}")
    print(f"Total predictors: {len(predictor_cols)}")
    print(f"\nPredictors:")
    for i, col in enumerate(predictor_cols, 1):
        print(f"  {i}. {col}")
    
    # Results dictionary
    results = {}
    
    # Loop through each domain
    for outcome_domain in outcome_domains:
        print(f"\n{'='*80}")
        print(f"OUTCOME DOMAIN: {outcome_domain}")
        print(f"{'='*80}")
        
        # Get outcome variable
        y = score_comparison[outcome_domain].copy()
        
        # Remove any rows with missing outcome
        valid_idx = y.notna()
        X_domain = X[valid_idx].copy()
        y_domain = y[valid_idx].copy()
        
        print(f"Valid observations: {len(y_domain)} / {len(y)}")
        
        if len(y_domain) < 20:
            print(f"WARNING: Too few observations ({len(y_domain)}) for {outcome_domain}. Skipping.")
            continue
        
        # ========== STEP 1: BOOTSTRAP STABILITY SELECTION ==========
        print(f"\n{'─'*80}")
        print(f"STEP 1: BOOTSTRAP STABILITY SELECTION (500 iterations)")
        print(f"{'─'*80}")
        print("Purpose: Identify predictors that are consistently selected across resampled datasets")
        
        n_bootstrap = 500
        feature_selection_counts = {col: 0 for col in predictor_cols}
        
        alphas = np.logspace(-4, 1, 100)
        
        for boot_idx in range(n_bootstrap):
            # Resample data with replacement
            X_boot, y_boot = resample(X_domain, y_domain, random_state=boot_idx)
            
            # Fit LASSO with CV
            lasso_cv = LassoCV(alphas=alphas, cv=10, random_state=42, max_iter=10000, n_jobs=-1)
            lasso_cv.fit(X_boot, y_boot)
            
            # Count selected features (non-zero coefficients)
            for i, col in enumerate(predictor_cols):
                if lasso_cv.coef_[i] != 0:
                    feature_selection_counts[col] += 1
            
            # Progress indicator
            if (boot_idx + 1) % 100 == 0:
                print(f"  Completed {boot_idx + 1}/{n_bootstrap} bootstrap iterations")
        
        # Calculate selection frequencies
        feature_selection_freq = {col: count / n_bootstrap * 100 for col, count in feature_selection_counts.items()}
        
        # Select stable features (selected in ≥50% of bootstrap samples)
        stable_features = [col for col, freq in feature_selection_freq.items() if freq >= 50]
        
        print(f"\n✓ Bootstrap Selection Frequencies:")
        sorted_features = sorted(feature_selection_freq.items(), key=lambda x: x[1], reverse=True)
        for feature, freq in sorted_features:
            selected = "✓" if freq >= 50 else "✗"
            print(f"  [{selected}] {feature:<40} {freq:>6.1f}%")
        
        print(f"\n✓ STABLE FEATURES IDENTIFIED: {len(stable_features)} / {len(predictor_cols)}")
        if len(stable_features) > 0:
            print(f"  Features: {', '.join(stable_features)}")
        else:
            print(f"  WARNING: No stable features identified (all <50% selection frequency)")
        
        # ========== STEP 2: NESTED CV USING ONLY STABLE FEATURES ==========
        print(f"\n{'─'*80}")
        print(f"STEP 2: NESTED 10-FOLD CV (using ONLY stable features)")
        print(f"{'─'*80}")
        print("Purpose: Get unbiased performance estimates using only the stable predictors")
        
        if len(stable_features) == 0:
            print("\nWARNING: Cannot perform nested CV - no stable features identified")
            print("Setting nested CV metrics to NaN")
            mean_nested_cv_r2 = np.nan
            mean_nested_cv_rmse = np.nan
            std_nested_cv_r2 = np.nan
            std_nested_cv_rmse = np.nan
            nested_cv_r2_scores = []
            nested_cv_rmse_scores = []
            nested_cv_predictions = []
            nested_cv_true_values = []
        else:
            # Use ONLY stable features
            X_stable = X_domain[stable_features].copy()
            
            print(f"Using {len(stable_features)} stable features for nested CV")
            
            # Outer loop: 10-fold CV for performance estimation
            outer_cv = KFold(n_splits=10, shuffle=True, random_state=42)
            
            # Inner loop: 10-fold CV for hyperparameter tuning
            inner_cv = KFold(n_splits=10, shuffle=True, random_state=42)
            
            # Store nested CV results
            nested_cv_r2_scores = []
            nested_cv_rmse_scores = []
            nested_cv_predictions = []
            nested_cv_true_values = []
            
            # Outer CV loop
            for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_stable), 1):
                X_train, X_test = X_stable.iloc[train_idx], X_stable.iloc[test_idx]
                y_train, y_test = y_domain.iloc[train_idx], y_domain.iloc[test_idx]
                
                # Inner CV: Tune lambda using LassoCV
                alphas = np.logspace(-4, 1, 100)
                lasso_cv = LassoCV(alphas=alphas, cv=inner_cv, random_state=42, max_iter=10000, n_jobs=-1)
                lasso_cv.fit(X_train, y_train)
                
                # Predict on outer test fold
                y_pred = lasso_cv.predict(X_test)
                
                # Calculate metrics
                r2 = r2_score(y_test, y_pred)
                rmse = np.sqrt(mean_squared_error(y_test, y_pred))
                
                nested_cv_r2_scores.append(r2)
                nested_cv_rmse_scores.append(rmse)
                nested_cv_predictions.extend(y_pred)
                nested_cv_true_values.extend(y_test)
                
                print(f"  Outer Fold {fold_idx}: R² = {r2:.4f}, RMSE = {rmse:.4f}, Best alpha = {lasso_cv.alpha_:.6f}")
            
            # Calculate mean nested CV performance
            mean_nested_cv_r2 = np.mean(nested_cv_r2_scores)
            mean_nested_cv_rmse = np.mean(nested_cv_rmse_scores)
            std_nested_cv_r2 = np.std(nested_cv_r2_scores)
            std_nested_cv_rmse = np.std(nested_cv_rmse_scores)
            
            print(f"\n✓ NESTED CV PERFORMANCE (unbiased estimates):")
            print(f"  Mean R²: {mean_nested_cv_r2:.4f} (SD: {std_nested_cv_r2:.4f})")
            print(f"  Mean RMSE: {mean_nested_cv_rmse:.4f} (SD: {std_nested_cv_rmse:.4f})")
        
        # ========== STEP 3: FIT FINAL LASSO MODEL USING ONLY STABLE FEATURES ==========
        print(f"\n{'─'*80}")
        print(f"STEP 3: FINAL LASSO MODEL ON FULL DATA (using ONLY stable features)")
        print(f"{'─'*80}")
        print("Purpose: Get coefficient estimates for reference (Step 4 will use OLS for interpretability)")
        
        if len(stable_features) == 0:
            print("\nWARNING: Cannot fit final model - no stable features identified")
            final_lasso_coefficients = pd.DataFrame({
                'Feature': predictor_cols,
                'Coefficient': [0.0] * len(predictor_cols),
                'Bootstrap_Selection_%': [feature_selection_freq[col] for col in predictor_cols],
                'Stable': [False] * len(predictor_cols)
            })
            best_alpha_final = np.nan
            apparent_r2 = np.nan
            apparent_rmse = np.nan
        else:
            # Use ONLY stable features
            X_stable_full = X_domain[stable_features].copy()
            
            # Fit final LASSO model on all data for coefficient estimates
            alphas = np.logspace(-4, 1, 100)
            lasso_cv_full = LassoCV(alphas=alphas, cv=10, random_state=42, max_iter=10000, n_jobs=-1)
            lasso_cv_full.fit(X_stable_full, y_domain)
            
            best_alpha_final = lasso_cv_full.alpha_
            
            # Calculate apparent performance (on training data - will be optimistic)
            y_pred_full = lasso_cv_full.predict(X_stable_full)
            apparent_r2 = r2_score(y_domain, y_pred_full)
            apparent_rmse = np.sqrt(mean_squared_error(y_domain, y_pred_full))
            
            # Create coefficient dataframe for ALL predictors
            coefficients_dict = {col: 0.0 for col in predictor_cols}  # Initialize all to 0
            for i, col in enumerate(stable_features):
                coefficients_dict[col] = lasso_cv_full.coef_[i]
            
            final_lasso_coefficients = pd.DataFrame({
                'Feature': predictor_cols,
                'Coefficient': [coefficients_dict[col] for col in predictor_cols],
                'Bootstrap_Selection_%': [feature_selection_freq[col] for col in predictor_cols],
                'Stable': [col in stable_features for col in predictor_cols]
            })
            final_lasso_coefficients = final_lasso_coefficients.sort_values('Bootstrap_Selection_%', ascending=False)
            
            print(f"\n✓ Final LASSO model fitted on {len(stable_features)} stable features")
            print(f"  Best alpha: {best_alpha_final:.6f}")
            print(f"  Apparent R² (optimistic): {apparent_r2:.4f}")
            print(f"  Apparent RMSE (optimistic): {apparent_rmse:.4f}")
            
            print(f"\n✓ Stable features and their LASSO coefficients:")
            for _, row in final_lasso_coefficients[final_lasso_coefficients['Stable'] == True].iterrows():
                print(f"  {row['Feature']:<40} Coef: {row['Coefficient']:>10.4f}  Selection: {row['Bootstrap_Selection_%']:>6.1f}%")
        
        # ========== STORE RESULTS ==========
        results[outcome_domain] = {
            # Data info
            'n_observations': len(y_domain),
            
            # Bootstrap results
            'feature_selection_freq': feature_selection_freq,
            'stable_features': stable_features,  # ← THIS WILL BE USED IN STEP 4
            'n_stable_features': len(stable_features),
            
            # Nested CV performance (UNBIASED - primary metric for reporting)
            'nested_cv_r2_mean': mean_nested_cv_r2,
            'nested_cv_r2_std': std_nested_cv_r2,
            'nested_cv_rmse_mean': mean_nested_cv_rmse,
            'nested_cv_rmse_std': std_nested_cv_rmse,
            'nested_cv_r2_scores': nested_cv_r2_scores,
            'nested_cv_rmse_scores': nested_cv_rmse_scores,
            'nested_cv_predictions': nested_cv_predictions,
            'nested_cv_true_values': nested_cv_true_values,
            
            # Final LASSO model (for reference only)
            'final_lasso_coefficients': final_lasso_coefficients,
            'final_lasso_best_alpha': best_alpha_final,
            'apparent_r2': apparent_r2,  # Will be optimistic
            'apparent_rmse': apparent_rmse  # Will be optimistic
        }
        
        print(f"\n{'='*80}")
        print(f"✓ COMPLETED: {outcome_domain}")
        print(f"{'='*80}")
    
    # ========== FINAL SUMMARY ==========
    print(f"\n{'='*80}")
    print("STEP 3 COMPLETE: NESTED LASSO WITH BOOTSTRAP STABILITY SELECTION")
    print(f"{'='*80}")
    
    # Summary table
    print("\nSUMMARY TABLE:")
    print("="*120)
    summary_rows = []
    for domain, res in results.items():
        summary_rows.append({
            'Domain': domain.replace('_12mo', ''),
            'N': res['n_observations'],
            'Stable Features': res['n_stable_features'],
            'Nested CV R² (mean±SD)': f"{res['nested_cv_r2_mean']:.4f}±{res['nested_cv_r2_std']:.4f}" if not np.isnan(res['nested_cv_r2_mean']) else "N/A",
            'Nested CV RMSE (mean±SD)': f"{res['nested_cv_rmse_mean']:.4f}±{res['nested_cv_rmse_std']:.4f}" if not np.isnan(res['nested_cv_rmse_mean']) else "N/A",
            'Apparent R² (optimistic)': f"{res['apparent_r2']:.4f}" if not np.isnan(res['apparent_r2']) else "N/A"
        })
    
    summary_df = pd.DataFrame(summary_rows)
    print(summary_df.to_string(index=False))
    print("="*120)
    
    print("\nNOTE:")
    print("  • Nested CV R²/RMSE = Unbiased performance estimates (use for reporting)")
    print("  • Apparent R²/RMSE = Optimistic estimates (use for comparison to show optimism)")
    print("  • Stable features will be used in Step 4 for final OLS regression")
    
    return results

# Run nested LASSO with bootstrap stability selection
lasso_results = perform_nested_lasso_cv_with_bootstrap(score_comparison)

# Save results
print("\n" + "="*80)
print("SAVING STEP 3 RESULTS...")
print("="*80)

for domain, res in lasso_results.items():
    domain_name = domain.replace('_12mo', '')
    
    # Save all coefficients with bootstrap frequencies
    res['final_lasso_coefficients'].to_csv(f"step3_lasso_coefficients_{domain_name}.csv", index=False)
    print(f"✓ Saved LASSO coefficients for {domain_name}")
    
    # Save stable features list (IMPORTANT: will be used in Step 4)
    if len(res['stable_features']) > 0:
        stable_df = pd.DataFrame({
            'Stable_Feature': res['stable_features'],
            'Bootstrap_Selection_%': [res['feature_selection_freq'][f] for f in res['stable_features']]
        })
        stable_df.to_csv(f"step3_stable_features_{domain_name}.csv", index=False)
        print(f"✓ Saved stable features for {domain_name} (will be used in Step 4)")
    else:
        print(f"⚠ No stable features for {domain_name}")
    
    # Save nested CV performance
    cv_performance = pd.DataFrame({
        'Metric': ['Nested_CV_R2_mean', 'Nested_CV_R2_std', 'Nested_CV_RMSE_mean', 'Nested_CV_RMSE_std',
                    'Apparent_R2', 'Apparent_RMSE', 'N_observations', 'N_stable_features'],
        'Value': [res['nested_cv_r2_mean'], res['nested_cv_r2_std'], res['nested_cv_rmse_mean'], 
                    res['nested_cv_rmse_std'], res['apparent_r2'], res['apparent_rmse'],
                    res['n_observations'], res['n_stable_features']]
    })
    cv_performance.to_csv(f"step3_nested_cv_performance_{domain_name}.csv", index=False)
    print(f"✓ Saved nested CV performance for {domain_name}")

print("="*80)
print("✓ STEP 3 COMPLETE - Ready for Step 4 (Final OLS Regression)")
print("="*80)

STEP 3: NESTED LASSO REGRESSION WITH BOOTSTRAP STABILITY SELECTION

Total patients: 108
Total predictors: 12

Predictors:
  1. fact_e_total_Baseline_std
  2. bmi_std
  3. age_diagnosis_std
  4. cci_category_numeric
  5. gender
  6. smoking_collapsed_imputed
  7. alcohol_collapsed_imputed
  8. level_tumor_collapsed_imputed
  9. path_histology_collapsed_imputed
  10. path_esoph_stage_collapsed_imputed
  11. surgery_approach_imputed
  12. neotx_collapsed_imputed

OUTCOME DOMAIN: fact_e_total_12mo
Valid observations: 108 / 108

────────────────────────────────────────────────────────────────────────────────
STEP 1: BOOTSTRAP STABILITY SELECTION (500 iterations)
────────────────────────────────────────────────────────────────────────────────
Purpose: Identify predictors that are consistently selected across resampled datasets
  Completed 100/500 bootstrap iterations
  Completed 200/500 bootstrap iterations
  Completed 300/500 bootstrap iterations
  Completed 400/500 bootstrap iterations
  C

In [6]:
lasso_results

{'fact_e_total_12mo': {'n_observations': 108,
  'feature_selection_freq': {'fact_e_total_Baseline_std': 97.0,
   'bmi_std': 61.6,
   'age_diagnosis_std': 79.2,
   'cci_category_numeric': 47.199999999999996,
   'gender': 37.2,
   'smoking_collapsed_imputed': 59.4,
   'alcohol_collapsed_imputed': 53.400000000000006,
   'level_tumor_collapsed_imputed': 52.0,
   'path_histology_collapsed_imputed': 60.4,
   'path_esoph_stage_collapsed_imputed': 68.4,
   'surgery_approach_imputed': 44.0,
   'neotx_collapsed_imputed': 76.0},
  'stable_features': ['fact_e_total_Baseline_std',
   'bmi_std',
   'age_diagnosis_std',
   'smoking_collapsed_imputed',
   'alcohol_collapsed_imputed',
   'level_tumor_collapsed_imputed',
   'path_histology_collapsed_imputed',
   'path_esoph_stage_collapsed_imputed',
   'neotx_collapsed_imputed'],
  'n_stable_features': 9,
  'nested_cv_r2_mean': -0.14263921329582704,
  'nested_cv_r2_std': 0.24255659724402007,
  'nested_cv_rmse_mean': 25.52366209180811,
  'nested_cv_rmse_

# Final regression modeling

In [12]:
with open('lasso_results.pkl', 'rb') as f:  # Load
    lasso_results = pickle.load(f)
lasso_results

{'fact_e_total_12mo': {'n_observations': 108,
  'feature_selection_freq': {'fact_e_total_Baseline_std': 97.0,
   'bmi_std': 61.6,
   'age_diagnosis_std': 79.2,
   'cci_category_numeric': 47.199999999999996,
   'gender': 37.2,
   'smoking_collapsed_imputed': 59.4,
   'alcohol_collapsed_imputed': 53.400000000000006,
   'level_tumor_collapsed_imputed': 52.0,
   'path_histology_collapsed_imputed': 60.4,
   'path_esoph_stage_collapsed_imputed': 68.4,
   'surgery_approach_imputed': 44.0,
   'neotx_collapsed_imputed': 76.0},
  'stable_features': ['fact_e_total_Baseline_std',
   'bmi_std',
   'age_diagnosis_std',
   'smoking_collapsed_imputed',
   'alcohol_collapsed_imputed',
   'level_tumor_collapsed_imputed',
   'path_histology_collapsed_imputed',
   'path_esoph_stage_collapsed_imputed',
   'neotx_collapsed_imputed'],
  'n_stable_features': 9,
  'nested_cv_r2_mean': -0.14263921329582704,
  'nested_cv_r2_std': 0.24255659724402007,
  'nested_cv_rmse_mean': 25.52366209180811,
  'nested_cv_rmse_

In [3]:
score_comparison=pd.read_csv("score_comparison_preprocessed.csv", index_col='id')
score_comparison

,baseline_type,fact_e_total_Baseline,fact_e_total_12mo,pwb_subscale_score_Baseline,pwb_subscale_score_12mo,swb_subscale_score_Baseline,swb_subscale_score_12mo,ewb_subscale_score_Baseline,ewb_subscale_score_12mo,fwb_subscale_score_Baseline,...,path_histology_collapsed_imputed,path_esoph_stage_collapsed_imputed,surgery_approach_imputed,neotx_collapsed_imputed,age_diagnosis,age_diagnosis_std,gender,cci_total,cci_total_std,cci_category_numeric
id,,,,,,,,,,,,,,,,,,,,,
32,baseline_arm_1,117.5,159.0,24.5,27.0,22.0,28.0,6.0,24.0,19.0,...,0,4,0,1,29.722718,-3.924010,1.0,2.0,-1.324120,1
76,preoperative_arm_1,95.0,97.0,7.0,7.0,23.0,21.0,14.0,11.0,15.0,...,1,2,1,1,57.597350,-0.759442,1.0,4.0,0.044349,2
125,baseline_arm_1,119.5,129.0,21.0,22.0,17.5,21.0,21.0,19.0,8.0,...,1,3,0,1,64.110830,-0.019975,1.0,4.0,0.044349,2
160,baseline_arm_1,119.0,152.0,21.0,28.0,20.0,22.0,21.0,23.0,20.0,...,1,3,0,1,50.804602,-1.530613,1.0,4.0,0.044349,2
198,baseline_arm_1,133.0,139.0,25.0,26.5,13.0,18.0,17.0,23.0,21.0,...,3,2,1,1,66.229970,0.220608,2.0,4.0,0.044349,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1566,preoperative_arm_1,150.0,107.0,23.0,19.0,24.0,19.0,18.0,15.0,18.0,...,1,3,1,1,69.876863,0.634635,1.0,4.0,0.044349,2
1571,baseline_arm_1,153.0,104.0,26.0,16.0,28.0,28.0,16.0,15.0,25.0,...,0,2,0,1,79.245980,1.698298,1.0,6.0,1.412817,3
1572,baseline_arm_1,141.0,115.0,27.0,24.0,13.0,21.0,19.0,18.0,20.0,...,3,3,0,1,60.529648,-0.426542,1.0,4.0,0.044349,2


In [28]:
def perform_final_ols_regression(score_comparison, lasso_results):
    """
    Step 4: Final OLS Regression Modeling
    
    Fit OLS regression using ONLY the stable features identified in Step 3.
    This provides interpretable coefficients, confidence intervals, and p-values
    for nomogram construction and clinical interpretation.
    
    Args:
        score_comparison: DataFrame with all preprocessed data
        lasso_results: Dictionary from Step 3 containing stable features for each domain
    
    Returns:
        Dictionary with OLS results for each domain including:
        - Regression coefficients with 95% CI and p-values
        - Model performance metrics (R², RMSE, Adjusted R²)
        - Fitted model object
        - Residuals for diagnostics
    """
    import statsmodels.api as sm
    from sklearn.metrics import mean_squared_error, r2_score
    import scipy.stats as stats
    
    print("="*80)
    print("STEP 4: FINAL OLS REGRESSION MODELING")
    print("="*80)
    print("Purpose: Fit interpretable OLS models using stable features from Step 3")
    print("Output: Coefficients (β), 95% CI, p-values for nomogram construction")
    
    # Results dictionary
    ols_results = {}
    
    # Loop through each domain
    for outcome_domain, lasso_res in lasso_results.items():
        print(f"\n{'='*80}")
        print(f"OUTCOME DOMAIN: {outcome_domain}")
        print(f"{'='*80}")
        
        # Get stable features from Step 3
        stable_features = lasso_res['stable_features']
        
        if len(stable_features) == 0:
            print("\n⚠ WARNING: No stable features identified in Step 3")
            print("   Skipping OLS regression for this domain")
            print("   Consider using a less stringent bootstrap threshold or more data")
            
            ols_results[outcome_domain] = {
                'n_observations': lasso_res['n_observations'],
                'stable_features': [],
                'n_stable_features': 0,
                'coefficients_table': None,
                'model_performance': None,
                'model': None,
                'skipped': True,
                'skip_reason': 'No stable features'
            }
            continue
        
        print(f"\nUsing {len(stable_features)} stable features from Step 3:")
        for i, feature in enumerate(stable_features, 1):
            selection_freq = lasso_res['feature_selection_freq'][feature]
            print(f"  {i}. {feature:<40} (Bootstrap selection: {selection_freq:.1f}%)")
        
        # Get outcome variable
        y = score_comparison[outcome_domain].copy()
        
        # Get predictor variables (ONLY stable features)
        X = score_comparison[stable_features].copy()
        
        # Remove any rows with missing outcome
        valid_idx = y.notna()
        X_clean = X[valid_idx].copy()
        y_clean = y[valid_idx].copy()
        
        print(f"\nValid observations: {len(y_clean)} / {len(y)}")
        
        # Add intercept
        X_clean_with_const = sm.add_constant(X_clean)
        
        # ========== FIT OLS REGRESSION ==========
        print(f"\n{'─'*80}")
        print("FITTING OLS REGRESSION MODEL")
        print(f"{'─'*80}")
        
        # Fit OLS model
        ols_model = sm.OLS(y_clean, X_clean_with_const).fit()
        
        # Get predictions
        y_pred = ols_model.predict(X_clean_with_const)
        
        # Calculate performance metrics
        r2 = r2_score(y_clean, y_pred)
        rmse = np.sqrt(mean_squared_error(y_clean, y_pred))
        adj_r2 = ols_model.rsquared_adj
        aic = ols_model.aic
        bic = ols_model.bic
        
        print(f"\n✓ MODEL PERFORMANCE (Apparent - optimistic):")
        print(f"  R²: {r2:.4f}")
        print(f"  Adjusted R²: {adj_r2:.4f}")
        print(f"  RMSE: {rmse:.4f}")
        print(f"  AIC: {aic:.2f}")
        print(f"  BIC: {bic:.2f}")
        print(f"  F-statistic: {ols_model.fvalue:.4f}")
        print(f"  Prob (F-statistic): {ols_model.f_pvalue:.4e}")
        
        # ========== EXTRACT COEFFICIENTS, CI, P-VALUES ==========
        print(f"\n{'─'*80}")
        print("REGRESSION COEFFICIENTS (for Nomogram)")
        print(f"{'─'*80}")
        
        # Create coefficients table
        coef_table = pd.DataFrame({
            'Feature': ['Intercept'] + stable_features,
            'Coefficient': ols_model.params.values,
            'Std_Error': ols_model.bse.values,
            'CI_Lower_95': ols_model.conf_int()[0].values,
            'CI_Upper_95': ols_model.conf_int()[1].values,
            't_statistic': ols_model.tvalues.values,
            'p_value': ols_model.pvalues.values
        })
        
        # Add significance stars
        def add_significance_stars(p):
            if p < 0.001:
                return '***'
            elif p < 0.01:
                return '**'
            elif p < 0.05:
                return '*'
            else:
                return ''
        
        coef_table['Significance'] = coef_table['p_value'].apply(add_significance_stars)
        
        # Display coefficients
        print("\nCoefficient | 95% CI | p-value")
        print("-" * 80)
        for idx, row in coef_table.iterrows():
            print(f"{row['Feature']:<40} "
                    f"β={row['Coefficient']:>8.4f} "
                    f"[{row['CI_Lower_95']:>7.4f}, {row['CI_Upper_95']:>7.4f}] "
                    f"p={row['p_value']:>7.4f} {row['Significance']}")
        
        print("\nSignificance: *** p<0.001, ** p<0.01, * p<0.05")
        
        # ========== MODEL DIAGNOSTICS ==========
        print(f"\n{'─'*80}")
        print("MODEL DIAGNOSTICS")
        print(f"{'─'*80}")
        
        # Calculate residuals
        residuals = ols_model.resid
        
        # Normality test (Shapiro-Wilk)
        if len(residuals) <= 5000:  # Shapiro-Wilk has sample size limit
            shapiro_stat, shapiro_p = stats.shapiro(residuals)
            print(f"Shapiro-Wilk test (normality): W={shapiro_stat:.4f}, p={shapiro_p:.4f}")
        else:
            print(f"Shapiro-Wilk test: Skipped (sample size > 5000)")
        
        # Durbin-Watson (independence)
        dw_stat = sm.stats.stattools.durbin_watson(residuals)
        print(f"Durbin-Watson (independence): {dw_stat:.4f} (ideal: ~2.0)")
        
        # Variance Inflation Factor (multicollinearity)
        if len(stable_features) > 1:
            from statsmodels.stats.outliers_influence import variance_inflation_factor
            vif_data = pd.DataFrame()
            vif_data["Feature"] = stable_features
            vif_data["VIF"] = [variance_inflation_factor(X_clean.values, i) for i in range(len(stable_features))]
            print(f"\nVariance Inflation Factors (multicollinearity):")
            for _, row in vif_data.iterrows():
                vif_concern = " (CONCERN: VIF > 10)" if row['VIF'] > 10 else ""
                print(f"  {row['Feature']:<40} VIF={row['VIF']:>8.4f}{vif_concern}")
        
        # ========== COMPARE WITH NESTED CV (from Step 3) ==========
        print(f"\n{'─'*80}")
        print("COMPARISON: OLS (Apparent) vs Nested CV (Unbiased)")
        print(f"{'─'*80}")
        
        nested_cv_r2 = lasso_res['nested_cv_r2_mean']
        nested_cv_rmse = lasso_res['nested_cv_rmse_mean']
        
        if not np.isnan(nested_cv_r2):
            optimism_r2 = r2 - nested_cv_r2
            optimism_rmse = rmse - nested_cv_rmse
            
            print(f"\nMetric                    OLS (Apparent)    Nested CV (Unbiased)    Optimism")
            print("-" * 80)
            print(f"R²                        {r2:>8.4f}          {nested_cv_r2:>8.4f}             {optimism_r2:>+8.4f}")
            print(f"RMSE                      {rmse:>8.4f}          {nested_cv_rmse:>8.4f}             {optimism_rmse:>+8.4f}")
            
            print(f"\nNote: Optimism = Apparent - Unbiased")
            print(f"      Positive optimism indicates overfitting on training data")
        else:
            print("\nNested CV results not available for comparison")
        
        # ========== STORE RESULTS ==========
        ols_results[outcome_domain] = {
            # Data info
            'n_observations': len(y_clean),
            'stable_features': stable_features,
            'n_stable_features': len(stable_features),
            
            # Coefficients table (for nomogram and reporting)
            'coefficients_table': coef_table,
            
            # Model performance
            'model_performance': {
                'r2': r2,
                'adjusted_r2': adj_r2,
                'rmse': rmse,
                'aic': aic,
                'bic': bic,
                'f_statistic': ols_model.fvalue,
                'f_pvalue': ols_model.f_pvalue
            },
            
            # Model object
            'model': ols_model,
            
            # Predictions and residuals
            'predictions': y_pred,
            'residuals': residuals,
            'y_true': y_clean,
            
            # Comparison with nested CV
            'nested_cv_r2': nested_cv_r2,
            'nested_cv_rmse': nested_cv_rmse,
            'optimism_r2': r2 - nested_cv_r2 if not np.isnan(nested_cv_r2) else np.nan,
            'optimism_rmse': rmse - nested_cv_rmse if not np.isnan(nested_cv_rmse) else np.nan,
            
            # Diagnostics
            'durbin_watson': dw_stat,
            
            'skipped': False
        }
        
        print(f"\n{'='*80}")
        print(f"✓ COMPLETED: {outcome_domain}")
        print(f"{'='*80}")
    
    # ========== FINAL SUMMARY ==========
    print(f"\n{'='*80}")
    print("STEP 4 COMPLETE: FINAL OLS REGRESSION MODELING")
    print(f"{'='*80}")
    
    # Summary table
    print("\nSUMMARY TABLE:")
    print("="*120)
    summary_rows = []
    for domain, res in ols_results.items():
        if res['skipped']:
            summary_rows.append({
                'Domain': domain.replace('_12mo', ''),
                'N': res['n_observations'],
                'Features': 0,
                'OLS R²': 'N/A',
                'OLS RMSE': 'N/A',
                'Nested CV R²': 'N/A',
                'Optimism (R²)': 'N/A'
            })
        else:
            nested_cv_r2_str = f"{res['nested_cv_r2']:.4f}" if not np.isnan(res['nested_cv_r2']) else 'N/A'
            optimism_str = f"{res['optimism_r2']:+.4f}" if not np.isnan(res['optimism_r2']) else 'N/A'
            
            summary_rows.append({
                'Domain': domain.replace('_12mo', ''),
                'N': res['n_observations'],
                'Features': res['n_stable_features'],
                'OLS R²': f"{res['model_performance']['r2']:.4f}",
                'OLS RMSE': f"{res['model_performance']['rmse']:.4f}",
                'Nested CV R²': nested_cv_r2_str,
                'Optimism (R²)': optimism_str
            })
    
    summary_df = pd.DataFrame(summary_rows)
    print(summary_df.to_string(index=False))
    print("="*120)
    
    print("\nNOTE:")
    print("  • OLS R²/RMSE = Apparent performance (optimistic)")
    print("  • Nested CV R² = Unbiased performance from Step 3 (use for reporting)")
    print("  • Optimism = OLS - Nested CV (shows degree of overfitting)")
    print("  • Coefficients will be used for nomogram construction (Step 5)")
    
    return ols_results

# Run final OLS regression
ols_results = perform_final_ols_regression(score_comparison, lasso_results)

# Save results
print("\n" + "="*80)
print("SAVING STEP 4 RESULTS...")
print("="*80)

for domain, res in ols_results.items():
    domain_name = domain.replace('_12mo', '')
    
    if res['skipped']:
        print(f"⚠ Skipped {domain_name} - no stable features")
        continue
    
    # Save coefficients table (IMPORTANT: for nomogram)
    res['coefficients_table'].to_csv(f"step4_ols_coefficients_{domain_name}.csv", index=False)
    print(f"✓ Saved OLS coefficients for {domain_name} (for nomogram)")
    
    # Save model performance comparison
    performance_df = pd.DataFrame({
        'Metric': ['N_observations', 'N_features', 'OLS_R2', 'OLS_Adjusted_R2', 'OLS_RMSE', 
                    'OLS_AIC', 'OLS_BIC', 'Nested_CV_R2', 'Nested_CV_RMSE', 
                    'Optimism_R2', 'Optimism_RMSE'],
        'Value': [
            res['n_observations'],
            res['n_stable_features'],
            res['model_performance']['r2'],
            res['model_performance']['adjusted_r2'],
            res['model_performance']['rmse'],
            res['model_performance']['aic'],
            res['model_performance']['bic'],
            res['nested_cv_r2'],
            res['nested_cv_rmse'],
            res['optimism_r2'],
            res['optimism_rmse']
        ]
    })
    performance_df.to_csv(f"step4_ols_performance_{domain_name}.csv", index=False)
    print(f"✓ Saved OLS performance comparison for {domain_name}")
    
    # Save predictions and residuals for diagnostics
    diagnostics_df = pd.DataFrame({
        'y_true': res['y_true'],
        'y_predicted': res['predictions'],
        'residuals': res['residuals']
    })
    diagnostics_df.to_csv(f"step4_ols_diagnostics_{domain_name}.csv", index=False)
    print(f"✓ Saved OLS diagnostics for {domain_name}")

print("="*80)
print("✓ STEP 4 COMPLETE - Ready for Step 5 (Nomogram Construction)")
print("="*80)

STEP 4: FINAL OLS REGRESSION MODELING
Purpose: Fit interpretable OLS models using stable features from Step 3
Output: Coefficients (β), 95% CI, p-values for nomogram construction

OUTCOME DOMAIN: fact_e_total_12mo

Using 9 stable features from Step 3:
  1. fact_e_total_Baseline_std                (Bootstrap selection: 97.0%)
  2. bmi_std                                  (Bootstrap selection: 61.6%)
  3. age_diagnosis_std                        (Bootstrap selection: 79.2%)
  4. smoking_collapsed_imputed                (Bootstrap selection: 59.4%)
  5. alcohol_collapsed_imputed                (Bootstrap selection: 53.4%)
  6. level_tumor_collapsed_imputed            (Bootstrap selection: 52.0%)
  7. path_histology_collapsed_imputed         (Bootstrap selection: 60.4%)
  8. path_esoph_stage_collapsed_imputed       (Bootstrap selection: 68.4%)
  9. neotx_collapsed_imputed                  (Bootstrap selection: 76.0%)

Valid observations: 108 / 108

─────────────────────────────────────────

In [29]:
ols_results

{'fact_e_total_12mo': {'n_observations': 108,
  'stable_features': ['fact_e_total_Baseline_std',
   'bmi_std',
   'age_diagnosis_std',
   'smoking_collapsed_imputed',
   'alcohol_collapsed_imputed',
   'level_tumor_collapsed_imputed',
   'path_histology_collapsed_imputed',
   'path_esoph_stage_collapsed_imputed',
   'neotx_collapsed_imputed'],
  'n_stable_features': 9,
  'coefficients_table':                               Feature  Coefficient  Std_Error  CI_Lower_95  \
  0                           Intercept   119.551757   9.994945    99.717115   
  1           fact_e_total_Baseline_std     6.007555   2.553481     0.940254   
  2                             bmi_std     1.944158   2.656021    -3.326629   
  3                   age_diagnosis_std     3.924228   2.582627    -1.200911   
  4           smoking_collapsed_imputed    -8.023516   5.287895   -18.517172   
  5           alcohol_collapsed_imputed     8.075283   5.462651    -2.765170   
  6       level_tumor_collapsed_imputed     3.

# Nomogram construction

In [32]:
# LOAD
with open('ols_results.pkl', 'rb') as f:
    ols_results = pickle.load(f)
ols_results

{'fact_e_total_12mo': {'n_observations': 108,
  'stable_features': ['fact_e_total_Baseline_std',
   'bmi_std',
   'age_diagnosis_std',
   'smoking_collapsed_imputed',
   'alcohol_collapsed_imputed',
   'level_tumor_collapsed_imputed',
   'path_histology_collapsed_imputed',
   'path_esoph_stage_collapsed_imputed',
   'neotx_collapsed_imputed'],
  'n_stable_features': 9,
  'coefficients_table':                               Feature  Coefficient  Std_Error  CI_Lower_95  \
  0                           Intercept   119.551757   9.994945    99.717115   
  1           fact_e_total_Baseline_std     6.007555   2.553481     0.940254   
  2                             bmi_std     1.944158   2.656021    -3.326629   
  3                   age_diagnosis_std     3.924228   2.582627    -1.200911   
  4           smoking_collapsed_imputed    -8.023516   5.287895   -18.517172   
  5           alcohol_collapsed_imputed     8.075283   5.462651    -2.765170   
  6       level_tumor_collapsed_imputed     3.